In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
OUT_DIR = "/kaggle/working/results/phase_outputs"

WINDOW_SIZE = 16
STRIDE = 8
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42
TOP_CATEGORIES = 10

os.makedirs(OUT_DIR, exist_ok=True)

header = pd.read_csv(DATA_PATH, nrows=0)
all_cols = list(header.columns)
lower_to_orig = {c.lower(): c for c in all_cols}


def pick_column(exact_names=None, contains=None, exclude=None):
    exact_names = exact_names or []
    contains = contains or []
    exclude = set(exclude or [])

    for name in exact_names:
        key = name.lower()
        if key in lower_to_orig and lower_to_orig[key] not in exclude:
            return lower_to_orig[key]

    for token in contains:
        token = token.lower()
        for col in all_cols:
            col_lower = col.lower()
            if token in col_lower and col not in exclude:
                return col

    return None


sender_col = pick_column(
    exact_names=["sender", "senderPseudo", "sender_id", "senderId", "vehicle_id", "vehicleId", "node_id", "nodeId", "src"],
    contains=["sender", "vehicle", "node", "src"]
)

time_col = pick_column(
    exact_names=["sendTime", "timestamp", "time", "timestep", "time_step", "ts"],
    contains=["sendtime", "timestamp", "time", "timestep", "time_step", "ts"]
)

type_col = pick_column(
    exact_names=["type", "attack_family", "attackFamily", "attack_type", "attackType", "label"],
    contains=["attack_family", "attacktype", "attack_type", "type", "label"]
)

class_col = pick_column(
    exact_names=["class", "binary_label", "is_malicious", "malicious"],
    contains=["binary", "malicious", "class"]
)

scenario_candidates = []
for name in [
    "scenario", "scenario_id", "scenarioId", "run", "run_id", "runId", "trace", "trace_id",
    "traceId", "simrun", "sim_run", "source_file", "sourceFile", "filename", "file", "seed",
    "repetition", "repeat", "density", "traffic", "map", "layout"
]:
    col = pick_column(exact_names=[name], contains=[name])
    if col is not None and col not in scenario_candidates and col not in [sender_col, time_col, type_col, class_col]:
        scenario_candidates.append(col)

usecols = []
for col in [sender_col, time_col, type_col, class_col] + scenario_candidates:
    if col is not None and col not in usecols:
        usecols.append(col)

df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)


def infer_binary_label(series):
    s = series.copy()
    out = pd.Series(np.nan, index=s.index, dtype="float64")

    numeric = pd.to_numeric(s, errors="coerce")
    out.loc[numeric.notna()] = (numeric.loc[numeric.notna()] != 0).astype(float)

    text = s.astype(str).str.strip().str.lower()
    benign = text.str.contains(
        r"\b(benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True,
        na=False
    )
    malicious = text.str.contains(
        r"\b(malicious|attack|attacker|spoof|bogus|sybil|dos|false|fake|ghost|replay|wormhole|jam)\b",
        regex=True,
        na=False
    )

    out.loc[benign & out.isna()] = 0.0
    out.loc[malicious & out.isna()] = 1.0
    return out


label_parts = []
if class_col is not None:
    label_parts.append(infer_binary_label(df[class_col]))
if type_col is not None and type_col != class_col:
    label_parts.append(infer_binary_label(df[type_col]))

if label_parts:
    label_values = pd.concat(label_parts, axis=1).max(axis=1, skipna=True)
    df["is_malicious"] = label_values.fillna(0).astype(bool)
else:
    df["is_malicious"] = False


def normalize_text_family(series):
    values = series.astype(str).str.strip().str.lower()
    values = values.str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
    values = values.replace({
        "": "",
        "nan": "",
        "none": "",
        "null": "",
        "0": "",
        "1": "unknown_malicious",
        "attack": "unknown_malicious",
        "attacker": "unknown_malicious",
        "malicious": "unknown_malicious",
        "true": "unknown_malicious",
        "false": "",
        "benign": "",
        "normal": "",
        "genuine": "",
        "legit": ""
    })
    return values


family_a = normalize_text_family(df[type_col]) if type_col is not None else pd.Series("", index=df.index)
family_b = normalize_text_family(df[class_col]) if class_col is not None else pd.Series("", index=df.index)

attack_family = family_a.where(family_a.ne(""), family_b)
attack_family = attack_family.where(df["is_malicious"], "benign")
attack_family = attack_family.replace("", "unknown_malicious")
df["attack_family"] = attack_family

scenario_source = None
if scenario_candidates:
    scenario_id = df[scenario_candidates[0]].astype(str).fillna("NA")
    for col in scenario_candidates[1:]:
        scenario_id = scenario_id + "|" + df[col].astype(str).fillna("NA")
    df["scenario_id"] = scenario_id
    scenario_source = "explicit_columns"
elif sender_col is not None:
    df["scenario_id"] = "sender_fallback|" + df[sender_col].astype(str).fillna("UNK")
    scenario_source = "sender_fallback"
else:
    df["scenario_id"] = "global_0"
    scenario_source = "global_fallback"

df["scenario_id"] = df["scenario_id"].astype("category")

scenario_stats = df.groupby("scenario_id", observed=True)["is_malicious"].agg(
    n_messages="size",
    n_malicious="sum"
).reset_index()

scenario_stats["n_benign"] = scenario_stats["n_messages"] - scenario_stats["n_malicious"]

mal_df = df.loc[df["is_malicious"], ["scenario_id", "attack_family"]].copy()
if len(mal_df) > 0:
    dominant_family = (
        mal_df.groupby(["scenario_id", "attack_family"], observed=True)
        .size()
        .reset_index(name="n")
        .sort_values(["scenario_id", "n"], ascending=[True, False])
        .drop_duplicates("scenario_id")
        .set_index("scenario_id")["attack_family"]
    )
    scenario_stats["dominant_family"] = scenario_stats["scenario_id"].map(dominant_family).fillna("benign")
else:
    scenario_stats["dominant_family"] = "benign"

scenario_stats["stratum"] = scenario_stats["dominant_family"].astype(str)
stratum_counts = scenario_stats["stratum"].value_counts()
scenario_stats.loc[scenario_stats["stratum"].map(stratum_counts).fillna(0).lt(2), "stratum"] = "other"


def deterministic_split(ids):
    split_result = {}
    for sid in ids:
        h = int(hashlib.md5(str(sid).encode("utf-8")).hexdigest(), 16) % 10000
        if h < int(TRAIN_RATIO * 10000):
            split_result[sid] = "train"
        elif h < int((TRAIN_RATIO + VAL_RATIO) * 10000):
            split_result[sid] = "validation"
        else:
            split_result[sid] = "test"
    return split_result


scenario_ids = scenario_stats["scenario_id"].astype(str).tolist()
split_map = {}

try:
    if len(scenario_stats) >= 3:
        stratify_1 = (
            scenario_stats["stratum"]
            if scenario_stats["stratum"].nunique() > 1 and scenario_stats["stratum"].value_counts().min() >= 2
            else None
        )

        train_ids, temp_ids = train_test_split(
            scenario_stats["scenario_id"].astype(str),
            test_size=(1.0 - TRAIN_RATIO),
            random_state=SEED,
            stratify=stratify_1
        )

        temp_df = scenario_stats[scenario_stats["scenario_id"].astype(str).isin(temp_ids)].copy()
        val_share_of_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

        stratify_2 = (
            temp_df["stratum"]
            if temp_df["stratum"].nunique() > 1 and temp_df["stratum"].value_counts().min() >= 2
            else None
        )

        val_ids, test_ids = train_test_split(
            temp_df["scenario_id"].astype(str),
            test_size=(1.0 - val_share_of_temp),
            random_state=SEED,
            stratify=stratify_2
        )

        split_map.update({sid: "train" for sid in train_ids})
        split_map.update({sid: "validation" for sid in val_ids})
        split_map.update({sid: "test" for sid in test_ids})
    else:
        split_map = deterministic_split(scenario_ids)

except Exception:
    split_map = deterministic_split(scenario_ids)

df["split"] = df["scenario_id"].astype(str).map(split_map).fillna("train")

window_group_cols = ["scenario_id"]
if sender_col is not None and scenario_source != "sender_fallback":
    window_group_cols.append(sender_col)

window_counts = (
    df.groupby(window_group_cols + ["split"], observed=True)
    .size()
    .reset_index(name="n_messages_group")
)

window_counts["n_windows"] = np.where(
    window_counts["n_messages_group"] >= WINDOW_SIZE,
    ((window_counts["n_messages_group"] - WINDOW_SIZE) // STRIDE) + 1,
    0
)

split_windows = (
    window_counts.groupby("split", observed=True)["n_windows"]
    .sum()
    .reindex(["train", "validation", "test"], fill_value=0)
)

attack_counts = (
    df.loc[df["is_malicious"], "attack_family"]
    .value_counts(dropna=False)
    .rename_axis("attack_family")
    .reset_index(name="count")
)

if len(attack_counts) > TOP_CATEGORIES:
    top_part = attack_counts.iloc[:TOP_CATEGORIES].copy()
    other_count = attack_counts.iloc[TOP_CATEGORIES:]["count"].sum()
    plot_categories = pd.concat(
        [top_part, pd.DataFrame([{"attack_family": "other", "count": other_count}])],
        ignore_index=True
    )
else:
    plot_categories = attack_counts.copy()

split_summary = (
    df.groupby("split", observed=True)["is_malicious"]
    .agg(n_messages="size", n_malicious="sum")
    .reindex(["train", "validation", "test"], fill_value=0)
    .reset_index()
)

split_summary["n_benign"] = split_summary["n_messages"] - split_summary["n_malicious"]
split_summary["n_scenarios"] = split_summary["split"].map(
    pd.Series(split_map).value_counts().reindex(["train", "validation", "test"], fill_value=0)
).values

split_summary["n_windows"] = split_summary["split"].map(split_windows).fillna(0).astype(int)

split_summary["n_attack_families"] = split_summary["split"].map(
    df.loc[df["is_malicious"]]
    .groupby("split", observed=True)["attack_family"]
    .nunique()
    .reindex(["train", "validation", "test"], fill_value=0)
).fillna(0).astype(int)

split_summary["malicious_ratio"] = np.where(
    split_summary["n_messages"] > 0,
    split_summary["n_malicious"] / split_summary["n_messages"],
    0.0
)

overall_summary = pd.DataFrame([{
    "split": "overall",
    "n_scenarios": int(df["scenario_id"].astype(str).nunique()),
    "n_windows": int(window_counts["n_windows"].sum()),
    "n_messages": int(len(df)),
    "n_benign": int((~df["is_malicious"]).sum()),
    "n_malicious": int(df["is_malicious"].sum()),
    "n_attack_families": int(df.loc[df["is_malicious"], "attack_family"].nunique()),
    "malicious_ratio": float(df["is_malicious"].mean())
}])

summary_table = pd.concat([split_summary, overall_summary], ignore_index=True)
summary_table = summary_table[[
    "split",
    "n_scenarios",
    "n_windows",
    "n_messages",
    "n_benign",
    "n_malicious",
    "n_attack_families",
    "malicious_ratio"
]]

summary_table.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False)

scenario_manifest = scenario_stats.copy()
scenario_manifest["split"] = scenario_manifest["scenario_id"].astype(str).map(split_map)
scenario_manifest.to_csv(os.path.join(OUT_DIR, "split_manifest.csv"), index=False)

attack_counts.to_csv(os.path.join(OUT_DIR, "category_distribution.csv"), index=False)

metadata = {
    "data_path": DATA_PATH,
    "out_dir": OUT_DIR,
    "selected_columns": {
        "sender_col": sender_col,
        "time_col": time_col,
        "type_col": type_col,
        "class_col": class_col,
        "scenario_candidates": scenario_candidates
    },
    "scenario_id_strategy": scenario_source,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "validation": VAL_RATIO,
        "test": TEST_RATIO
    },
    "n_rows": int(len(df)),
    "n_scenarios": int(df["scenario_id"].astype(str).nunique()),
    "n_attack_families": int(df.loc[df["is_malicious"], "attack_family"].nunique())
}

with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig = plt.figure(figsize=(17, 5.5), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.5, 1.25])

ax1 = fig.add_subplot(gs[0, 0])
binary_counts = pd.Series({
    "Benign": int((~df["is_malicious"]).sum()),
    "Malicious": int(df["is_malicious"].sum())
})
bars1 = ax1.bar(binary_counts.index, binary_counts.values)
ax1.set_title("Class Distribution")
ax1.set_ylabel("Count")

for bar, val in zip(bars1, binary_counts.values):
    pct = 100.0 * val / len(df) if len(df) else 0.0
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{val:,}\n({pct:.1f}%)",
        ha="center",
        va="bottom"
    )

ax2 = fig.add_subplot(gs[0, 1])
plot_df = plot_categories.sort_values("count", ascending=True)
bars2 = ax2.barh(plot_df["attack_family"], plot_df["count"])
ax2.set_title("Category Distribution")
ax2.set_xlabel("Count")

for bar, val in zip(bars2, plot_df["count"].values):
    ax2.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {val:,}",
        va="center"
    )

ax3 = fig.add_subplot(gs[0, 2])
plot_split = split_summary.set_index("split").reindex(["train", "validation", "test"])
x = np.arange(len(plot_split))

ax3.bar(x, plot_split["n_benign"].values, label="Benign")
ax3.bar(
    x,
    plot_split["n_malicious"].values,
    bottom=plot_split["n_benign"].values,
    label="Malicious"
)

ax3.set_xticks(x)
ax3.set_xticklabels(["Train", "Validation", "Test"])
ax3.set_title("Data Split Overview")
ax3.set_ylabel("Count")
ax3.legend(frameon=False)

totals = plot_split["n_messages"].values
for i, (_, row) in enumerate(plot_split.iterrows()):
    ax3.text(
        i,
        totals[i],
        f"{int(row['n_messages']):,}\nS={int(row['n_scenarios'])}, W={int(row['n_windows']):,}",
        ha="center",
        va="bottom"
    )

fig.suptitle("Data Overview", y=1.03, fontsize=14)

fig_png = os.path.join(OUT_DIR, "overview_figure.png")
fig_pdf = os.path.join(OUT_DIR, "overview_figure.pdf")

plt.savefig(fig_png, bbox_inches="tight")
plt.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
OUT_DIR = "/kaggle/working/results/2_Main Detection Results"

SAMPLE_ROWS_FOR_SCHEMA = 200000
MAX_FEATURES = 20
TRAIN_CAP = 350000
VAL_CAP = 120000
TEST_CAP = None
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)

def downcast_numeric_df(df):
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)
    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b", regex=True, na=False)
    malicious = txt.str.contains(r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b", regex=True, na=False)

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def evaluate_label_candidate(series, colname):
    y = infer_binary_label(series)
    info = {
        "column": colname,
        "dtype": str(series.dtype),
        "sample_non_null": int(series.notna().sum()),
        "sample_unique": int(series.nunique(dropna=True)),
        "top_values": " | ".join(series.astype(str).value_counts(dropna=False).head(10).index.astype(str).tolist())
    }
    if y is None or y.nunique(dropna=True) < 2:
        info.update({
            "usable": 0,
            "pos_ratio": np.nan,
            "score": -1.0
        })
        return info, None

    pos_ratio = float(y.mean())
    balance_score = max(0.0, 1.0 - abs(pos_ratio - 0.5) * 2.0)
    score = 10.0 + balance_score
    if 0.001 < pos_ratio < 0.999:
        score += 3.0
    if 0.01 < pos_ratio < 0.99:
        score += 2.0

    info.update({
        "usable": 1,
        "pos_ratio": pos_ratio,
        "score": score
    })
    return info, y

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(int)
    n = len(y)
    if (max_rows is None) or (n <= max_rows):
        return np.arange(n)
    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)
    rs = np.random.RandomState(seed)
    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)
    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def ensure_2col_proba(prob):
    prob = np.asarray(prob)
    if prob.ndim == 1:
        prob = np.column_stack([1.0 - prob, prob])
    if prob.shape[1] == 1:
        prob = np.column_stack([1.0 - prob[:, 0], prob[:, 0]])
    prob = np.clip(prob, 1e-7, 1 - 1e-7)
    prob = prob / prob.sum(axis=1, keepdims=True)
    return prob

def ece_score(y_true, proba, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    proba = ensure_2col_proba(proba)
    pred = proba.argmax(axis=1)
    conf = proba.max(axis=1)
    correct = (pred == y_true).astype(np.float32)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (conf > bins[i]) & (conf <= bins[i + 1]) if i > 0 else (conf >= bins[i]) & (conf <= bins[i + 1])
        if mask.sum() == 0:
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += (mask.mean()) * abs(acc_bin - conf_bin)
    return float(ece)

def macro_auprc_binary(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    proba = ensure_2col_proba(proba)
    y_onehot = np.zeros((len(y_true), 2), dtype=np.int8)
    y_onehot[np.arange(len(y_true)), y_true] = 1
    return float(average_precision_score(y_onehot, proba, average="macro"))

def choose_best_threshold(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos)
    thresholds = np.linspace(0.05, 0.95, 37)
    best_thr = 0.50
    best_score = -1.0
    for thr in thresholds:
        pred = (prob_pos >= thr).astype(np.int8)
        score = f1_score(y_true, pred, average="macro", zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr

stage_bar = tqdm(total=8, desc="overall", position=0)

sample_df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS_FOR_SCHEMA, low_memory=False)
all_cols = sample_df.columns.tolist()

selected_columns = {"sender_col": None, "time_col": None, "type_col": None, "class_col": None, "scenario_candidates": []}
meta_path = os.path.join(BENCH_DIR, "benchmark_metadata.json")
if os.path.exists(meta_path):
    with open(meta_path, "r") as f:
        meta = json.load(f)
    selected_columns.update(meta.get("selected_columns", {}))

candidate_label_cols = []
for c in [selected_columns.get("class_col"), selected_columns.get("type_col")]:
    if c is not None and c in all_cols and c not in candidate_label_cols:
        candidate_label_cols.append(c)

for c in all_cols:
    cl = c.lower()
    if any(tok in cl for tok in ["label", "class", "attack", "malicious", "benign", "type"]):
        if c not in candidate_label_cols:
            candidate_label_cols.append(c)

diag_rows = []
best_label_col = None
best_score = -1.0
for col in tqdm(candidate_label_cols, desc="label scan", position=1, leave=False):
    info, y_try = evaluate_label_candidate(sample_df[col], col)
    diag_rows.append(info)
    if info["score"] > best_score:
        best_score = info["score"]
        best_label_col = col

diag_df = pd.DataFrame(diag_rows).sort_values(["usable", "score"], ascending=[False, False])
diag_df.to_csv(os.path.join(OUT_DIR, "label_candidate_diagnostics.csv"), index=False)

if best_label_col is None or best_score < 0:
    raise ValueError("No usable binary label column was inferred. Check label_candidate_diagnostics.csv")

stage_bar.update(1)

exclude_cols = set(candidate_label_cols)
exclude_cols.update([c for c in [
    selected_columns.get("sender_col"),
    selected_columns.get("time_col")
] if c is not None])
exclude_cols.update(selected_columns.get("scenario_candidates", []))

numeric_quality = []
for c in tqdm(all_cols, desc="feature scan", position=1, leave=False):
    if c in exclude_cols:
        continue
    x = pd.to_numeric(sample_df[c], errors="coerce")
    valid_ratio = float(x.notna().mean())
    nunq = int(x.nunique(dropna=True))
    if valid_ratio >= 0.90 and nunq > 1:
        var = float(x.var(skipna=True))
        numeric_quality.append((c, valid_ratio, nunq, var))

feat_df = pd.DataFrame(numeric_quality, columns=["column", "valid_ratio", "nunique", "variance"])
feat_df = feat_df.sort_values(["variance", "valid_ratio", "nunique"], ascending=[False, False, False])
feature_cols = feat_df["column"].head(MAX_FEATURES).tolist()
feat_df.to_csv(os.path.join(OUT_DIR, "feature_scan_diagnostics.csv"), index=False)

if len(feature_cols) == 0:
    raise ValueError("No usable numeric feature columns were found")

stage_bar.update(1)

usecols = sorted(set(feature_cols + candidate_label_cols + [
    c for c in [
        selected_columns.get("sender_col"),
        selected_columns.get("time_col")
    ] if c is not None
] + selected_columns.get("scenario_candidates", [])))

df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)
stage_bar.update(1)

df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[best_label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Chosen label column '{best_label_col}' did not yield a valid binary target on full data")

df["target"] = y.astype(np.int8)
df = df[df["split"].isin(["train", "validation", "test"])].copy()

X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")
X = downcast_numeric_df(X)

split_counts = df.groupby("split")["target"].agg(["size", "sum"]).reset_index()
split_counts["neg"] = split_counts["size"] - split_counts["sum"]
split_counts.to_csv(os.path.join(OUT_DIR, "split_label_distribution.csv"), index=False)

train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

y_train_full = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
y_val_full = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
y_test_full = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)

if len(np.unique(y_train_full)) < 2 or len(np.unique(y_val_full)) < 2 or len(np.unique(y_test_full)) < 2:
    raise ValueError("At least one split has only one class. Check split_label_distribution.csv and label_candidate_diagnostics.csv")

train_idx_local = stratified_cap_indices(y_train_full, TRAIN_CAP, seed=SEED)
val_idx_local = stratified_cap_indices(y_val_full, VAL_CAP, seed=SEED + 1)
test_idx_local = stratified_cap_indices(y_test_full, TEST_CAP, seed=SEED + 2)

X_train = X.loc[train_mask].iloc[train_idx_local].reset_index(drop=True)
y_train = y_train_full[train_idx_local]
X_val = X.loc[val_mask].iloc[val_idx_local].reset_index(drop=True)
y_val = y_val_full[val_idx_local]
X_test = X.loc[test_mask].iloc[test_idx_local].reset_index(drop=True)
y_test = y_test_full[test_idx_local]

stage_bar.update(1)

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

stage_bar.update(1)

models = [
    ("Dummy-Prior", DummyClassifier(strategy="prior", random_state=SEED), "raw"),
    ("GaussianNB", GaussianNB(), "raw"),
    ("SGD-LogLoss", SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=20, tol=1e-3, class_weight="balanced", random_state=SEED), "scaled"),
    ("LogReg-SAGA", LogisticRegression(solver="saga", penalty="l2", C=1.0, max_iter=60, class_weight="balanced", n_jobs=-1, random_state=SEED), "scaled"),
    ("HistGB", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED), "raw"),
]

results = []
threshold_rows = []

for name, model, mode in tqdm(models, desc="fit/eval models", position=1):
    Xt_train = X_train_scaled if mode == "scaled" else X_train_imp
    Xt_val = X_val_scaled if mode == "scaled" else X_val_imp
    Xt_test = X_test_scaled if mode == "scaled" else X_test_imp

    clf = clone(model)

    t0 = time.perf_counter()
    clf.fit(Xt_train, y_train)
    fit_sec = time.perf_counter() - t0

    val_proba = ensure_2col_proba(clf.predict_proba(Xt_val))
    thr = choose_best_threshold(y_val, val_proba[:, 1])

    t1 = time.perf_counter()
    test_proba = ensure_2col_proba(clf.predict_proba(Xt_test))
    infer_sec = time.perf_counter() - t1
    y_pred = (test_proba[:, 1] >= thr).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    far = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0

    row = {
        "method": name,
        "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "macro_auprc": float(macro_auprc_binary(y_test, test_proba)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "false_alarm_rate": far,
        "ece": float(ece_score(y_test, test_proba, n_bins=15)),
        "latency_us_per_sample": float((infer_sec / len(Xt_test)) * 1e6),
        "fit_seconds": float(fit_sec),
        "threshold": float(thr),
        "n_test_samples": int(len(Xt_test))
    }
    results.append(row)

    threshold_rows.append({
        "method": name,
        "threshold": float(thr),
        "fit_seconds": float(fit_sec)
    })

stage_bar.update(1)

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(OUT_DIR, "table2_main_detection_results_across_all_compared_methods.csv"), index=False)
pd.DataFrame(threshold_rows).to_csv(os.path.join(OUT_DIR, "model_thresholds_and_fit_time.csv"), index=False)

stage_bar.update(1)

plot_df = results_df.copy()
metrics_for_plot = ["macro_f1", "macro_auprc", "balanced_accuracy", "false_alarm_rate"]
metric_titles = ["Macro F1", "Macro AUPRC", "Balanced Accuracy", "False Alarm Rate"]

x = np.arange(len(plot_df))
width = 0.18

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, ax = plt.subplots(figsize=(16, 6))
for i, metric in enumerate(metrics_for_plot):
    ax.bar(x + (i - 1.5) * width, plot_df[metric].values, width=width, label=metric_titles[i])

ax.set_xticks(x)
ax.set_xticklabels(plot_df["method"].tolist(), rotation=20, ha="right")
ax.set_ylim(0, 1.02)
ax.set_ylabel("Score")
ax.set_title("Figure 2. Overall Model Comparison on Main Metrics")
ax.legend(frameon=False, ncol=2)
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig_png = os.path.join(OUT_DIR, "figure2_overall_model_comparison_on_main_metrics.png")
fig_pdf = os.path.join(OUT_DIR, "figure2_overall_model_comparison_on_main_metrics.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

stage_bar.update(1)
stage_bar.close()

summary = {
    "data_path": DATA_PATH,
    "benchmark_dir": BENCH_DIR,
    "output_dir": OUT_DIR,
    "chosen_label_column": best_label_col,
    "feature_columns": feature_cols,
    "train_rows_used": int(len(X_train)),
    "validation_rows_used": int(len(X_val)),
    "test_rows_used": int(len(X_test)),
    "n_models": int(len(models))
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(fig_png)
print(fig_pdf)
print(os.path.join(OUT_DIR, "table2_main_detection_results_across_all_compared_methods.csv"))
print(os.path.join(OUT_DIR, "label_candidate_diagnostics.csv"))
print(os.path.join(OUT_DIR, "feature_scan_diagnostics.csv"))
print(os.path.join(OUT_DIR, "split_label_distribution.csv"))
print()
print(results_df)

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
OUT_DIR = "/kaggle/working/results/3_Per Attack Family Analysis"

SAMPLE_ROWS_FOR_SCAN = 200000
TRAIN_CAP = 350000
VAL_CAP = 120000
TEST_CAP = None
MIN_SUPPORT_PER_FAMILY = 20
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def normalize_attack_family(series, y_binary=None):
    x = series.astype(str).str.strip().str.lower()
    x = x.str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()

    replace_map = {
        "": "",
        "nan": "",
        "none": "",
        "null": "",
        "false": "",
        "true": "unknown_malicious",
        "0": "",
        "1": "unknown_malicious",
        "benign": "benign",
        "normal": "benign",
        "genuine": "benign",
        "clean": "benign",
        "safe": "benign",
        "baseline": "benign",
        "malicious": "unknown_malicious",
        "attack": "unknown_malicious",
        "attacker": "unknown_malicious",
    }
    x = x.replace(replace_map)

    if y_binary is not None:
        y_binary = pd.Series(y_binary, index=series.index)
        x = x.where(y_binary.astype(int) == 1, "benign")
        x = x.where(~((y_binary.astype(int) == 1) & (x == "")), "unknown_malicious")

    return x

def choose_best_threshold_pos_f1(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)
    thresholds = np.linspace(0.05, 0.95, 37)
    best_thr = 0.50
    best_score = -1.0
    for thr in thresholds:
        pred = (prob_pos >= thr).astype(np.int8)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(np.int8)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def evaluate_family_candidate(series, y_binary):
    fam = normalize_attack_family(series, y_binary)
    fam_mal = fam[y_binary == 1]
    fam_mal = fam_mal[(fam_mal != "") & (fam_mal != "unknown_malicious")]

    unique_count = int(fam_mal.nunique(dropna=True))
    non_empty_ratio = float((fam_mal != "").mean()) if len(fam_mal) > 0 else 0.0
    top_vals = " | ".join(fam_mal.value_counts().head(10).index.astype(str).tolist())

    score = unique_count + 0.1 * non_empty_ratio
    return {
        "column": series.name,
        "usable": int(unique_count >= 1),
        "unique_malicious_families": unique_count,
        "non_empty_ratio_on_malicious": non_empty_ratio,
        "score": score,
        "top_families": top_vals
    }

overall_bar = tqdm(total=9, desc="overall", position=0)

with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

sample_df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS_FOR_SCAN, low_memory=False)
all_cols = sample_df.columns.tolist()

family_candidate_cols = []
for c in all_cols:
    cl = c.lower()
    if any(tok in cl for tok in ["attack", "family", "type", "class", "label"]):
        family_candidate_cols.append(c)

family_candidate_cols = list(dict.fromkeys(family_candidate_cols))
overall_bar.update(1)

y_sample = infer_binary_label(sample_df[label_col])
if y_sample is None or y_sample.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

family_diag_rows = []
for col in tqdm(family_candidate_cols, desc="family column scan", position=1, leave=False):
    info = evaluate_family_candidate(sample_df[col], y_sample)
    family_diag_rows.append(info)

family_diag_df = pd.DataFrame(family_diag_rows).sort_values(
    ["unique_malicious_families", "score"], ascending=[False, False]
)
family_diag_df.to_csv(os.path.join(OUT_DIR, "family_column_diagnostics.csv"), index=False)

best_family_col = family_diag_df.iloc[0]["column"]
overall_bar.update(1)


needed_cols = set(feature_cols + [label_col, best_family_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)


df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

df["target"] = infer_binary_label(df[label_col]).astype(np.int8)
df["attack_family"] = normalize_attack_family(df[best_family_col], df["target"])

df = df[df["split"].isin(["train", "validation", "test"])].copy()

X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")

y = df["target"].to_numpy(dtype=np.int8)

family_counts_all = (
    df.loc[df["target"] == 1, "attack_family"]
    .value_counts()
    .rename_axis("attack_family")
    .reset_index(name="count")
)
family_counts_all.to_csv(os.path.join(OUT_DIR, "attack_family_counts_full.csv"), index=False)

overall_bar.update(1)

train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

y_train_full = y[train_mask]
y_val_full = y[val_mask]
y_test_full = y[test_mask]

train_idx_local = stratified_cap_indices(y_train_full, TRAIN_CAP, seed=SEED)
val_idx_local = stratified_cap_indices(y_val_full, VAL_CAP, seed=SEED + 1)
test_idx_local = stratified_cap_indices(y_test_full, TEST_CAP, seed=SEED + 2)

X_train = X.loc[train_mask].iloc[train_idx_local].reset_index(drop=True)
y_train = y_train_full[train_idx_local]

X_val = X.loc[val_mask].iloc[val_idx_local].reset_index(drop=True)
y_val = y_val_full[val_idx_local]
fam_val = df.loc[val_mask, "attack_family"].iloc[val_idx_local].reset_index(drop=True)

X_test = X.loc[test_mask].iloc[test_idx_local].reset_index(drop=True)
y_test = y_test_full[test_idx_local]
fam_test = df.loc[test_mask, "attack_family"].iloc[test_idx_local].reset_index(drop=True)

overall_bar.update(1)

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.08,
    max_iter=120,
    random_state=SEED
)

t0 = time.perf_counter()
model.fit(X_train_imp, y_train)
fit_sec = time.perf_counter() - t0

val_prob = model.predict_proba(X_val_imp)[:, 1]
test_prob = model.predict_proba(X_test_imp)[:, 1]

overall_bar.update(1)


families = (
    fam_test[y_test == 1]
    .value_counts()
    .rename_axis("attack_family")
    .reset_index(name="support_test")
)

families = families[
    (families["attack_family"] != "unknown_malicious") &
    (families["attack_family"] != "benign") &
    (families["support_test"] >= MIN_SUPPORT_PER_FAMILY)
].copy()

table3_rows = []
curve_data = []
threshold_rows = []

for fam in tqdm(families["attack_family"].tolist(), desc="per-family metrics", position=1, leave=False):
    val_subset = (fam_val.eq("benign") | fam_val.eq(fam)).to_numpy()
    test_subset = (fam_test.eq("benign") | fam_test.eq(fam)).to_numpy()

    if val_subset.sum() == 0 or test_subset.sum() == 0:
        continue

    y_val_fam = (fam_val[val_subset] == fam).astype(np.int8).to_numpy()
    y_test_fam = (fam_test[test_subset] == fam).astype(np.int8).to_numpy()

    if y_val_fam.sum() < MIN_SUPPORT_PER_FAMILY or y_test_fam.sum() < MIN_SUPPORT_PER_FAMILY:
        continue

    prob_val_fam = val_prob[val_subset]
    prob_test_fam = test_prob[test_subset]

    thr_fam = choose_best_threshold_pos_f1(y_val_fam, prob_val_fam)
    pred_test_fam = (prob_test_fam >= thr_fam).astype(np.int8)

    prec_curve, rec_curve, _ = precision_recall_curve(y_test_fam, prob_test_fam)
    auprc = average_precision_score(y_test_fam, prob_test_fam)

    precision = precision_score(y_test_fam, pred_test_fam, zero_division=0)
    recall = recall_score(y_test_fam, pred_test_fam, zero_division=0)
    f1 = f1_score(y_test_fam, pred_test_fam, zero_division=0)

    table3_rows.append({
        "attack_family": fam,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auprc": float(auprc),
        "support": int(y_test_fam.sum()),
        "threshold": float(thr_fam)
    })

    threshold_rows.append({
        "attack_family": fam,
        "threshold": float(thr_fam),
        "val_support": int(y_val_fam.sum()),
        "test_support": int(y_test_fam.sum())
    })

    curve_data.append({
        "attack_family": fam,
        "precision_curve": prec_curve,
        "recall_curve": rec_curve,
        "auprc": float(auprc),
        "support": int(y_test_fam.sum())
    })

table3 = pd.DataFrame(table3_rows).sort_values(["f1", "auprc"], ascending=[False, False]).reset_index(drop=True)
table3.to_csv(os.path.join(OUT_DIR, "table3_class_wise_detection_performance_by_attack_family.csv"), index=False)

pd.DataFrame(threshold_rows).to_csv(os.path.join(OUT_DIR, "family_thresholds.csv"), index=False)
overall_bar.update(1)


plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, ax = plt.subplots(figsize=(10.5, 8))

for item in curve_data:
    ax.plot(
        item["recall_curve"],
        item["precision_curve"],
        linewidth=2,
        label=f"{item['attack_family']} (AUPRC={item['auprc']:.3f}, n={item['support']})"
    )

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.set_title("Figure 3. Per Attack Family Precision-Recall Curves")

if len(curve_data) <= 10:
    ax.legend(frameon=False, loc="lower left")
else:
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()

fig_png = os.path.join(OUT_DIR, "figure3_per_attack_family_precision_recall_curves.png")
fig_pdf = os.path.join(OUT_DIR, "figure3_per_attack_family_precision_recall_curves.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

overall_bar.update(1)
overall_bar.close()



In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
OUT_DIR = "/kaggle/working/results/4_Ablation Study of Model Components"

os.makedirs(OUT_DIR, exist_ok=True)


TRAIN_CAP = 350000
VAL_CAP = 120000
TEST_CAP = None
SEED = 42


def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(int)
    n = len(y)
    if (max_rows is None) or (n <= max_rows):
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def ensure_2col_proba(prob):
    prob = np.asarray(prob)
    if prob.ndim == 1:
        prob = np.column_stack([1.0 - prob, prob])
    if prob.shape[1] == 1:
        prob = np.column_stack([1.0 - prob[:, 0], prob[:, 0]])
    prob = np.clip(prob, 1e-7, 1 - 1e-7)
    prob = prob / prob.sum(axis=1, keepdims=True)
    return prob

def macro_auprc_binary(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(int)
    prob_2 = ensure_2col_proba(prob_pos)
    y_onehot = np.zeros((len(y_true), 2), dtype=np.int8)
    y_onehot[np.arange(len(y_true)), y_true] = 1
    return float(average_precision_score(y_onehot, prob_2, average="macro"))

def ece_score(y_true, prob_pos, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob_2 = ensure_2col_proba(prob_pos)
    pred = prob_2.argmax(axis=1)
    conf = prob_2.max(axis=1)
    correct = (pred == y_true).astype(np.float32)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == 0:
            mask = (conf >= bins[i]) & (conf <= bins[i + 1])
        else:
            mask = (conf > bins[i]) & (conf <= bins[i + 1])
        if mask.sum() == 0:
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += mask.mean() * abs(acc_bin - conf_bin)
    return float(ece)

def choose_best_threshold_macro_f1(y_true, prob_pos):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_thr = 0.50
    best_score = -1.0
    for thr in thresholds:
        pred = (prob_pos >= thr).astype(np.int8)
        score = f1_score(y_true, pred, average="macro", zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr

def evaluate_predictions(y_true, prob_pos, threshold, latency_us_per_sample):
    pred = (prob_pos >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    far = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0

    return {
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "macro_auprc": float(macro_auprc_binary(y_true, prob_pos)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "false_alarm_rate": far,
        "ece": float(ece_score(y_true, prob_pos)),
        "latency_us_per_sample": float(latency_us_per_sample)
    }

def build_feature_groups(feature_cols):
    temporal_tokens = [
        "time", "timestamp", "delta", "interval", "jitter", "window", "seq", "age",
        "speed", "heading", "acc", "accel", "velocity", "position", "pos", "x", "y", "z",
        "lat", "lon", "mean", "std", "var", "roll", "rate", "trend"
    ]
    graph_tokens = [
        "neighbor", "neigh", "graph", "degree", "adj", "edge", "hop", "cluster",
        "density", "local", "disagree", "context", "spatial", "distance", "dist",
        "relative", "consensus", "community"
    ]

    temporal = []
    graph = []

    for f in feature_cols:
        fl = f.lower()
        is_temporal = any(tok in fl for tok in temporal_tokens)
        is_graph = any(tok in fl for tok in graph_tokens)

        if is_temporal and not is_graph:
            temporal.append(f)
        elif is_graph and not is_temporal:
            graph.append(f)

    remaining = [f for f in feature_cols if f not in temporal and f not in graph]

    target_min = max(3, len(feature_cols) // 3)

    if len(temporal) < target_min:
        take = min(target_min - len(temporal), len(remaining))
        temporal += remaining[:take]
        remaining = remaining[take:]

    if len(graph) < target_min:
        take = min(target_min - len(graph), len(remaining))
        graph += remaining[:take]
        remaining = remaining[take:]

    if len(temporal) == 0:
        cut = max(1, len(feature_cols) // 2)
        temporal = feature_cols[:cut]

    if len(graph) == 0:
        cut = max(1, len(feature_cols) // 2)
        graph = feature_cols[cut:]
        if len(graph) == 0:
            graph = feature_cols[:]

    temporal = list(dict.fromkeys(temporal))
    graph = [f for f in graph if f not in temporal]
    if len(graph) == 0:
        graph = temporal[:]

    group_rows = []
    for f in feature_cols:
        if f in temporal and f in graph:
            grp = "both"
        elif f in temporal:
            grp = "temporal"
        elif f in graph:
            grp = "graph"
        else:
            grp = "shared_other"
        group_rows.append({"feature": f, "group": grp})

    return temporal, graph, pd.DataFrame(group_rows)

def fit_histgb(X_train, y_train):
    model = HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.08,
        max_iter=120,
        random_state=SEED
    )
    model.fit(X_train, y_train)
    return model

with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

overall_bar = tqdm(total=10, desc="overall", position=0)


needed_cols = set(feature_cols + [label_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)

df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

df["target"] = y.astype(np.int8)
df = df[df["split"].isin(["train", "validation", "test"])].copy()

X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")

overall_bar.update(1)

train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

y_train_full = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
y_val_full = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
y_test_full = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)

train_idx_local = stratified_cap_indices(y_train_full, TRAIN_CAP, seed=SEED)
val_idx_local = stratified_cap_indices(y_val_full, VAL_CAP, seed=SEED + 1)
test_idx_local = stratified_cap_indices(y_test_full, TEST_CAP, seed=SEED + 2)

X_train = X.loc[train_mask].iloc[train_idx_local].reset_index(drop=True)
y_train = y_train_full[train_idx_local]

X_val = X.loc[val_mask].iloc[val_idx_local].reset_index(drop=True)
y_val = y_val_full[val_idx_local]

X_test = X.loc[test_mask].iloc[test_idx_local].reset_index(drop=True)
y_test = y_test_full[test_idx_local]

overall_bar.update(1)


imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

overall_bar.update(1)

temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)
feature_group_df.to_csv(os.path.join(OUT_DIR, "feature_group_assignments.csv"), index=False)

feat_to_idx = {f: i for i, f in enumerate(feature_cols)}
temporal_idx = [feat_to_idx[f] for f in temporal_cols if f in feat_to_idx]
graph_idx = [feat_to_idx[f] for f in graph_cols if f in feat_to_idx]

X_train_temporal = X_train_imp[:, temporal_idx]
X_val_temporal = X_val_imp[:, temporal_idx]
X_test_temporal = X_test_imp[:, temporal_idx]

X_train_graph = X_train_imp[:, graph_idx]
X_val_graph = X_val_imp[:, graph_idx]
X_test_graph = X_test_imp[:, graph_idx]

overall_bar.update(1)


scaler_ssl = StandardScaler()
X_train_scaled = scaler_ssl.fit_transform(X_train_imp)
X_val_scaled = scaler_ssl.transform(X_val_imp)
X_test_scaled = scaler_ssl.transform(X_test_imp)

n_comp = max(1, min(8, X_train_scaled.shape[1]))
svd = TruncatedSVD(n_components=n_comp, random_state=SEED)

Z_train_ssl = svd.fit_transform(X_train_scaled)
Z_val_ssl = svd.transform(X_val_scaled)
Z_test_ssl = svd.transform(X_test_scaled)

overall_bar.update(1)

val_indices = np.arange(len(y_val))
idx_fuse, idx_rest = train_test_split(
    val_indices,
    test_size=0.60,
    random_state=SEED,
    stratify=y_val
)
idx_cal, idx_thr = train_test_split(
    idx_rest,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=y_val[idx_rest]
)

y_val_fuse = y_val[idx_fuse]
y_val_cal = y_val[idx_cal]
y_val_thr = y_val[idx_thr]

overall_bar.update(1)


branch_bar = tqdm(total=3, desc="fit branches", position=1, leave=False)

t0 = time.perf_counter()
model_temporal = fit_histgb(X_train_temporal, y_train)
fit_temporal_sec = time.perf_counter() - t0
branch_bar.update(1)

t0 = time.perf_counter()
model_graph = fit_histgb(X_train_graph, y_train)
fit_graph_sec = time.perf_counter() - t0
branch_bar.update(1)

t0 = time.perf_counter()
model_ssl = fit_histgb(Z_train_ssl, y_train)
fit_ssl_sec = time.perf_counter() - t0
branch_bar.update(1)
branch_bar.close()

# Predict validation/test once
p_val_temporal = model_temporal.predict_proba(X_val_temporal)[:, 1]
p_test_temporal = model_temporal.predict_proba(X_test_temporal)[:, 1]

p_val_graph = model_graph.predict_proba(X_val_graph)[:, 1]
p_test_graph = model_graph.predict_proba(X_test_graph)[:, 1]

p_val_ssl = model_ssl.predict_proba(Z_val_ssl)[:, 1]
p_test_ssl = model_ssl.predict_proba(Z_test_ssl)[:, 1]

overall_bar.update(1)

fusion_wo_ssl = LogisticRegression(solver="lbfgs", random_state=SEED)
fusion_wo_ssl.fit(
    np.column_stack([p_val_temporal[idx_fuse], p_val_graph[idx_fuse]]),
    y_val_fuse
)

fusion_full = LogisticRegression(solver="lbfgs", random_state=SEED)
fusion_full.fit(
    np.column_stack([p_val_temporal[idx_fuse], p_val_graph[idx_fuse], p_val_ssl[idx_fuse]]),
    y_val_fuse
)


raw_probs = {
    "Temporal only": {
        "val_cal": p_val_temporal[idx_cal],
        "val_thr": p_val_temporal[idx_thr],
        "test": p_test_temporal
    },
    "Graph only": {
        "val_cal": p_val_graph[idx_cal],
        "val_thr": p_val_graph[idx_thr],
        "test": p_test_graph
    },
    "Without self-supervision": {
        "val_cal": fusion_wo_ssl.predict_proba(np.column_stack([p_val_temporal[idx_cal], p_val_graph[idx_cal]]))[:, 1],
        "val_thr": fusion_wo_ssl.predict_proba(np.column_stack([p_val_temporal[idx_thr], p_val_graph[idx_thr]]))[:, 1],
        "test": fusion_wo_ssl.predict_proba(np.column_stack([p_test_temporal, p_test_graph]))[:, 1]
    },
    "Without calibration": {
        "val_cal": fusion_full.predict_proba(np.column_stack([p_val_temporal[idx_cal], p_val_graph[idx_cal], p_val_ssl[idx_cal]]))[:, 1],
        "val_thr": fusion_full.predict_proba(np.column_stack([p_val_temporal[idx_thr], p_val_graph[idx_thr], p_val_ssl[idx_thr]]))[:, 1],
        "test": fusion_full.predict_proba(np.column_stack([p_test_temporal, p_test_graph, p_test_ssl]))[:, 1]
    },
    "Without gated fusion": {
        "val_cal": (p_val_temporal[idx_cal] + p_val_graph[idx_cal] + p_val_ssl[idx_cal]) / 3.0,
        "val_thr": (p_val_temporal[idx_thr] + p_val_graph[idx_thr] + p_val_ssl[idx_thr]) / 3.0,
        "test": (p_test_temporal + p_test_graph + p_test_ssl) / 3.0
    },
    "Full model": {
        "val_cal": fusion_full.predict_proba(np.column_stack([p_val_temporal[idx_cal], p_val_graph[idx_cal], p_val_ssl[idx_cal]]))[:, 1],
        "val_thr": fusion_full.predict_proba(np.column_stack([p_val_temporal[idx_thr], p_val_graph[idx_thr], p_val_ssl[idx_thr]]))[:, 1],
        "test": fusion_full.predict_proba(np.column_stack([p_test_temporal, p_test_graph, p_test_ssl]))[:, 1]
    }
}


method_order = [
    "Temporal only",
    "Graph only",
    "Without self-supervision",
    "Without calibration",
    "Without gated fusion",
    "Full model"
]

results = []
eval_bar = tqdm(method_order, desc="evaluate ablations", position=1, leave=False)

for method in eval_bar:
    start_pred = time.perf_counter()

    if method == "Without calibration":
        prob_val_thr = raw_probs[method]["val_thr"]
        prob_test = raw_probs[method]["test"]
    else:
        calibrator = IsotonicRegression(out_of_bounds="clip")
        calibrator.fit(raw_probs[method]["val_cal"], y_val_cal)
        prob_val_thr = calibrator.transform(raw_probs[method]["val_thr"])
        prob_test = calibrator.transform(raw_probs[method]["test"])

    infer_sec = time.perf_counter() - start_pred
    latency_us_per_sample = (infer_sec / len(prob_test)) * 1e6

    threshold = choose_best_threshold_macro_f1(y_val_thr, prob_val_thr)
    metrics = evaluate_predictions(y_test, prob_test, threshold, latency_us_per_sample)

    results.append({
        "method": method,
        **metrics,
        "threshold": float(threshold)
    })

results_df = pd.DataFrame(results)

full_row = results_df.loc[results_df["method"] == "Full model"].iloc[0]

def safe_drop(full_val, cur_val):
    denom = full_val if abs(full_val) > 1e-12 else 1e-12
    return 100.0 * (full_val - cur_val) / denom

def safe_change(full_val, cur_val):
    denom = full_val if abs(full_val) > 1e-12 else 1e-12
    return 100.0 * (cur_val - full_val) / denom

results_df["macro_f1_drop_pct"] = results_df["macro_f1"].apply(lambda x: safe_drop(full_row["macro_f1"], x))
results_df["macro_auprc_drop_pct"] = results_df["macro_auprc"].apply(lambda x: safe_drop(full_row["macro_auprc"], x))
results_df["balanced_accuracy_drop_pct"] = results_df["balanced_accuracy"].apply(lambda x: safe_drop(full_row["balanced_accuracy"], x))
results_df["precision_drop_pct"] = results_df["precision"].apply(lambda x: safe_drop(full_row["precision"], x))
results_df["recall_drop_pct"] = results_df["recall"].apply(lambda x: safe_drop(full_row["recall"], x))

results_df["false_alarm_rate_change_pct"] = results_df["false_alarm_rate"].apply(lambda x: safe_change(full_row["false_alarm_rate"], x))
results_df["ece_change_pct"] = results_df["ece"].apply(lambda x: safe_change(full_row["ece"], x))
results_df["latency_change_pct"] = results_df["latency_us_per_sample"].apply(lambda x: safe_change(full_row["latency_us_per_sample"], x))

results_df = results_df.set_index("method").loc[method_order].reset_index()

table4 = results_df[[
    "method",
    "macro_f1",
    "macro_auprc",
    "precision",
    "recall",
    "balanced_accuracy",
    "false_alarm_rate",
    "ece",
    "latency_us_per_sample",
    "macro_f1_drop_pct",
    "macro_auprc_drop_pct",
    "balanced_accuracy_drop_pct",
    "precision_drop_pct",
    "recall_drop_pct",
    "false_alarm_rate_change_pct",
    "ece_change_pct",
    "latency_change_pct",
    "threshold"
]]

table4_path = os.path.join(OUT_DIR, "table4_ablation_results_for_architectural_components.csv")
table4.to_csv(table4_path, index=False)

overall_bar.update(1)


plot_df = results_df.copy()
metrics_for_plot = ["macro_f1", "macro_auprc", "balanced_accuracy", "false_alarm_rate"]
metric_titles = ["Macro F1", "Macro AUPRC", "Balanced Accuracy", "False Alarm Rate"]

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(plot_df))
width = 0.18

for i, metric in enumerate(metrics_for_plot):
    ax.bar(x + (i - 1.5) * width, plot_df[metric].values, width=width, label=metric_titles[i])

ax.set_xticks(x)
ax.set_xticklabels(plot_df["method"].tolist(), rotation=18, ha="right")
ax.set_ylim(0, 1.02)
ax.set_ylabel("Score")
ax.set_title("Figure 4. Ablation Study of Model Components")
ax.legend(frameon=False, ncol=2)
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig_png = os.path.join(OUT_DIR, "figure4_ablation_study_of_model_components.png")
fig_pdf = os.path.join(OUT_DIR, "figure4_ablation_study_of_model_components.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

overall_bar.update(1)
overall_bar.close()




In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import (
    f1_score,
    silhouette_score,
    davies_bouldin_score
)

warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
RESULTS3_DIR = "/kaggle/working/results/3_Per Attack Family Analysis"
RESULTS4_DIR = "/kaggle/working/results/4_Ablation Study of Model Components"
OUT_DIR = "/kaggle/working/results/5_Embedding Quality and Projection Analysis"

os.makedirs(OUT_DIR, exist_ok=True)


TRAIN_CAP = 300000
VAL_CAP = 100000
TEST_CAP = 150000

PROJ_SAMPLE_UMAP = 12000
PROJ_SAMPLE_TSNE = 6000
CLUSTER_SAMPLE = 5000
MIN_FAMILY_SUPPORT = 200
TOP_FAMILIES_FOR_FIGURE = 10
SEED = 42


def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def normalize_attack_family(series, y_binary=None):
    x = series.astype(str).str.strip().str.lower()
    x = x.str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()

    replace_map = {
        "": "",
        "nan": "",
        "none": "",
        "null": "",
        "false": "",
        "true": "unknown_malicious",
        "0": "",
        "1": "unknown_malicious",
        "benign": "benign",
        "normal": "benign",
        "genuine": "benign",
        "clean": "benign",
        "safe": "benign",
        "baseline": "benign",
        "malicious": "unknown_malicious",
        "attack": "unknown_malicious",
        "attacker": "unknown_malicious",
    }
    x = x.replace(replace_map)

    if y_binary is not None:
        y_binary = pd.Series(y_binary, index=series.index)
        x = x.where(y_binary.astype(int) == 1, "benign")
        x = x.where(~((y_binary.astype(int) == 1) & (x == "")), "unknown_malicious")

    return x

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(np.int8)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def safe_sample_indices(n, max_n, seed=42):
    if n <= max_n:
        return np.arange(n)
    rs = np.random.RandomState(seed)
    return np.sort(rs.choice(np.arange(n), size=max_n, replace=False))

def cluster_purity_score(y_true, cluster_labels):
    y_true = np.asarray(y_true)
    cluster_labels = np.asarray(cluster_labels)
    total = len(y_true)
    purity = 0.0
    for c in np.unique(cluster_labels):
        mask = cluster_labels == c
        if mask.sum() == 0:
            continue
        _, counts = np.unique(y_true[mask], return_counts=True)
        purity += counts.max()
    return float(purity / total)

def try_make_projector():
    try:
        import umap.umap_ as umap
        projector = umap.UMAP(
            n_neighbors=25,
            min_dist=0.15,
            n_components=2,
            metric="euclidean",
            random_state=SEED
        )
        return projector, "UMAP", PROJ_SAMPLE_UMAP
    except Exception:
        from sklearn.manifold import TSNE
        projector = TSNE(
            n_components=2,
            perplexity=30,
            learning_rate="auto",
            init="pca",
            random_state=SEED
        )
        return projector, "t-SNE", PROJ_SAMPLE_TSNE

def build_feature_groups(feature_cols):
    temporal_tokens = [
        "time", "timestamp", "delta", "interval", "jitter", "window", "seq", "age",
        "speed", "heading", "acc", "accel", "velocity", "position", "pos", "x", "y", "z",
        "lat", "lon", "mean", "std", "var", "roll", "rate", "trend"
    ]
    graph_tokens = [
        "neighbor", "neigh", "graph", "degree", "adj", "edge", "hop", "cluster",
        "density", "local", "disagree", "context", "spatial", "distance", "dist",
        "relative", "consensus", "community"
    ]

    temporal = []
    graph = []

    for f in feature_cols:
        fl = f.lower()
        is_temporal = any(tok in fl for tok in temporal_tokens)
        is_graph = any(tok in fl for tok in graph_tokens)

        if is_temporal and not is_graph:
            temporal.append(f)
        elif is_graph and not is_temporal:
            graph.append(f)

    remaining = [f for f in feature_cols if f not in temporal and f not in graph]

    def fill_group(group, target_min, donor_pool):
        need = max(0, target_min - len(group))
        take = donor_pool[:need]
        group.extend(take)
        donor_pool = donor_pool[need:]
        return group, donor_pool

    target_min = min(2, len(feature_cols))
    temporal, remaining = fill_group(temporal, target_min, remaining)
    graph, remaining = fill_group(graph, target_min, remaining)

    if len(temporal) == 0:
        temporal = feature_cols[:min(2, len(feature_cols))]
    if len(graph) == 0:
        leftover = [f for f in feature_cols if f not in temporal]
        if len(leftover) == 0:
            graph = temporal[:]
        else:
            graph = leftover[:min(2, len(leftover))]

    temporal = list(dict.fromkeys(temporal))
    graph = list(dict.fromkeys(graph))

    if len(graph) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in graph:
                graph.append(f)
            if len(graph) >= 2:
                break

    if len(temporal) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in temporal:
                temporal.append(f)
            if len(temporal) >= 2:
                break

    rows = []
    for f in feature_cols:
        if f in temporal and f in graph:
            grp = "both"
        elif f in temporal:
            grp = "temporal"
        elif f in graph:
            grp = "graph"
        else:
            grp = "shared_other"
        rows.append({"feature": f, "group": grp})

    return temporal, graph, pd.DataFrame(rows)

def fit_safe_latent_projection(X_train, X_val, X_test, max_comp=6, random_state=42, branch_name="branch"):
    n_features = X_train.shape[1]

    if n_features == 0:
        raise ValueError(f"{branch_name} has zero features")

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    if n_features == 1:
        info = {
            "branch": branch_name,
            "method": "identity_1d",
            "input_dim": int(n_features),
            "output_dim": 1
        }
        return X_train_s, X_val_s, X_test_s, info

    n_comp = max(1, min(max_comp, n_features - 1))
    reducer = TruncatedSVD(n_components=n_comp, random_state=random_state)

    Z_train = reducer.fit_transform(X_train_s)
    Z_val = reducer.transform(X_val_s)
    Z_test = reducer.transform(X_test_s)

    info = {
        "branch": branch_name,
        "method": "truncated_svd",
        "input_dim": int(n_features),
        "output_dim": int(n_comp)
    }
    return Z_train, Z_val, Z_test, info

# =========================================================
# Load metadata
# =========================================================
with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

family_col = None
results3_summary_path = os.path.join(RESULTS3_DIR, "run_summary.json")
if os.path.exists(results3_summary_path):
    with open(results3_summary_path, "r") as f:
        run3_meta = json.load(f)
    family_col = run3_meta.get("attack_family_column")

if family_col is None:
    fam_diag = pd.read_csv(os.path.join(RESULTS3_DIR, "family_column_diagnostics.csv"))
    family_col = fam_diag.iloc[0]["column"]

feature_group_path = os.path.join(RESULTS4_DIR, "feature_group_assignments.csv")
if os.path.exists(feature_group_path):
    feature_group_df = pd.read_csv(feature_group_path)
    temporal_cols = feature_group_df.loc[feature_group_df["group"].isin(["temporal", "both"]), "feature"].tolist()
    graph_cols = feature_group_df.loc[feature_group_df["group"].isin(["graph", "both"]), "feature"].tolist()

    if len(temporal_cols) < 2 or len(graph_cols) < 1:
        temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)
else:
    temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)

# =========================================================
# Load data
# =========================================================
overall_bar = tqdm(total=12, desc="overall", position=0)

needed_cols = set(feature_cols + [label_col, family_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)

# =========================================================
# Build split and labels
# =========================================================
df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

df["target"] = y.astype(np.int8)
df["attack_family"] = normalize_attack_family(df[family_col], df["target"])
df = df[df["split"].isin(["train", "validation", "test"])].copy()

overall_bar.update(1)

# =========================================================
# Numeric features
# =========================================================
X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")

overall_bar.update(1)

# =========================================================
# Split and cap
# =========================================================
train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

y_train_full = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
y_val_full = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
y_test_full = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)

train_idx_local = stratified_cap_indices(y_train_full, TRAIN_CAP, seed=SEED)
val_idx_local = stratified_cap_indices(y_val_full, VAL_CAP, seed=SEED + 1)
test_idx_local = stratified_cap_indices(y_test_full, TEST_CAP, seed=SEED + 2)

X_train = X.loc[train_mask].iloc[train_idx_local].reset_index(drop=True)
y_train = y_train_full[train_idx_local]
fam_train = df.loc[train_mask, "attack_family"].iloc[train_idx_local].reset_index(drop=True)

X_val = X.loc[val_mask].iloc[val_idx_local].reset_index(drop=True)
y_val = y_val_full[val_idx_local]
fam_val = df.loc[val_mask, "attack_family"].iloc[val_idx_local].reset_index(drop=True)

X_test = X.loc[test_mask].iloc[test_idx_local].reset_index(drop=True)
y_test = y_test_full[test_idx_local]
fam_test = df.loc[test_mask, "attack_family"].iloc[test_idx_local].reset_index(drop=True)

overall_bar.update(1)

# =========================================================
# Impute
# =========================================================
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

overall_bar.update(1)

# =========================================================
# Feature groups
# =========================================================
feat_to_idx = {f: i for i, f in enumerate(feature_cols)}
temporal_idx = [feat_to_idx[f] for f in temporal_cols if f in feat_to_idx]
graph_idx = [feat_to_idx[f] for f in graph_cols if f in feat_to_idx]

if len(temporal_idx) == 0:
    temporal_idx = list(range(min(2, len(feature_cols))))
if len(graph_idx) == 0:
    graph_idx = list(range(min(2, len(feature_cols))))

X_train_temporal = X_train_imp[:, temporal_idx]
X_val_temporal = X_val_imp[:, temporal_idx]
X_test_temporal = X_test_imp[:, temporal_idx]

X_train_graph = X_train_imp[:, graph_idx]
X_val_graph = X_val_imp[:, graph_idx]
X_test_graph = X_test_imp[:, graph_idx]

overall_bar.update(1)

# =========================================================
# Safe latent projections
# =========================================================
latent_bar = tqdm(total=3, desc="latent branches", position=1, leave=False)

Z_train_temp, Z_val_temp, Z_test_temp, info_temp = fit_safe_latent_projection(
    X_train_temporal, X_val_temporal, X_test_temporal,
    max_comp=6, random_state=SEED, branch_name="temporal"
)
latent_bar.update(1)

Z_train_graph, Z_val_graph, Z_test_graph, info_graph = fit_safe_latent_projection(
    X_train_graph, X_val_graph, X_test_graph,
    max_comp=6, random_state=SEED + 1, branch_name="graph"
)
latent_bar.update(1)

Z_train_ssl, Z_val_ssl, Z_test_ssl, info_ssl = fit_safe_latent_projection(
    X_train_imp, X_val_imp, X_test_imp,
    max_comp=8, random_state=SEED + 2, branch_name="ssl_proxy"
)
latent_bar.update(1)
latent_bar.close()

overall_bar.update(1)

# =========================================================
# Train branch models + gated fusion
# =========================================================
fit_bar = tqdm(total=4, desc="fit models", position=1, leave=False)

model_temp = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED)
model_graph = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 1)
model_ssl = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 2)

model_temp.fit(Z_train_temp, y_train)
fit_bar.update(1)

model_graph.fit(Z_train_graph, y_train)
fit_bar.update(1)

model_ssl.fit(Z_train_ssl, y_train)
fit_bar.update(1)

p_train_temp = model_temp.predict_proba(Z_train_temp)[:, 1]
p_val_temp = model_temp.predict_proba(Z_val_temp)[:, 1]
p_test_temp = model_temp.predict_proba(Z_test_temp)[:, 1]

p_train_graph = model_graph.predict_proba(Z_train_graph)[:, 1]
p_val_graph = model_graph.predict_proba(Z_val_graph)[:, 1]
p_test_graph = model_graph.predict_proba(Z_test_graph)[:, 1]

p_train_ssl = model_ssl.predict_proba(Z_train_ssl)[:, 1]
p_val_ssl = model_ssl.predict_proba(Z_val_ssl)[:, 1]
p_test_ssl = model_ssl.predict_proba(Z_test_ssl)[:, 1]

fusion = LogisticRegression(solver="lbfgs", max_iter=200, random_state=SEED)
fusion.fit(
    np.column_stack([p_val_temp, p_val_graph, p_val_ssl]),
    y_val
)
fit_bar.update(1)
fit_bar.close()

p_train_full = fusion.predict_proba(np.column_stack([p_train_temp, p_train_graph, p_train_ssl]))[:, 1]
p_val_full = fusion.predict_proba(np.column_stack([p_val_temp, p_val_graph, p_val_ssl]))[:, 1]
p_test_full = fusion.predict_proba(np.column_stack([p_test_temp, p_test_graph, p_test_ssl]))[:, 1]

overall_bar.update(1)

# =========================================================
# Final learned embeddings
# =========================================================
E_train = np.column_stack([
    Z_train_temp, Z_train_graph, Z_train_ssl,
    p_train_temp, p_train_graph, p_train_ssl, p_train_full
])
E_val = np.column_stack([
    Z_val_temp, Z_val_graph, Z_val_ssl,
    p_val_temp, p_val_graph, p_val_ssl, p_val_full
])
E_test = np.column_stack([
    Z_test_temp, Z_test_graph, Z_test_ssl,
    p_test_temp, p_test_graph, p_test_ssl, p_test_full
])

embed_scaler = StandardScaler()
E_train = embed_scaler.fit_transform(E_train)
E_val = embed_scaler.transform(E_val)
E_test = embed_scaler.transform(E_test)

overall_bar.update(1)

# =========================================================
# Table 5
# =========================================================
table_rows = []

# Binary
binary_eval_idx = safe_sample_indices(len(E_test), CLUSTER_SAMPLE, seed=SEED)
E_test_bin_eval = E_test[binary_eval_idx]
y_test_bin_eval = y_test[binary_eval_idx]

kmeans_bin = MiniBatchKMeans(n_clusters=2, random_state=SEED, batch_size=1024)
cluster_bin = kmeans_bin.fit_predict(E_test_bin_eval)

sil_bin = float(silhouette_score(
    E_test_bin_eval, y_test_bin_eval,
    sample_size=min(4000, len(E_test_bin_eval)),
    random_state=SEED
))
db_bin = float(davies_bouldin_score(E_test_bin_eval, y_test_bin_eval))
purity_bin = float(cluster_purity_score(y_test_bin_eval, cluster_bin))

probe_bin = LogisticRegression(solver="lbfgs", max_iter=300, random_state=SEED)
probe_bin.fit(E_train, y_train)
pred_bin = probe_bin.predict(E_test)
probe_bin_f1 = float(f1_score(y_test, pred_bin, average="macro", zero_division=0))

table_rows.append({
    "label_space": "benign_vs_malicious",
    "num_classes": 2,
    "num_samples_eval": int(len(E_test_bin_eval)),
    "silhouette_score": sil_bin,
    "davies_bouldin_index": db_bin,
    "cluster_purity": purity_bin,
    "linear_probe_macro_f1": probe_bin_f1
})

# Attack family
train_family_counts = fam_train[fam_train != "benign"].value_counts()
test_family_counts = fam_test[fam_test != "benign"].value_counts()

supported_families = sorted(set(
    train_family_counts[train_family_counts >= MIN_FAMILY_SUPPORT].index
).intersection(set(
    test_family_counts[test_family_counts >= MIN_FAMILY_SUPPORT].index
)))

fam_train_mask = fam_train.isin(supported_families).values
fam_test_mask = fam_test.isin(supported_families).values

if fam_train_mask.sum() > 0 and fam_test_mask.sum() > 0:
    E_train_fam = E_train[fam_train_mask]
    E_test_fam = E_test[fam_test_mask]
    y_train_fam = fam_train[fam_train_mask].astype(str).to_numpy()
    y_test_fam = fam_test[fam_test_mask].astype(str).to_numpy()

    if len(np.unique(y_train_fam)) >= 2 and len(np.unique(y_test_fam)) >= 2:
        le_fam = LabelEncoder()
        y_train_fam_enc = le_fam.fit_transform(y_train_fam)
        y_test_fam_enc = le_fam.transform(y_test_fam)

        fam_eval_idx = safe_sample_indices(len(E_test_fam), min(CLUSTER_SAMPLE, len(E_test_fam)), seed=SEED + 5)
        E_test_fam_eval = E_test_fam[fam_eval_idx]
        y_test_fam_eval = y_test_fam_enc[fam_eval_idx]

        n_classes_fam = len(np.unique(y_test_fam_eval))
        kmeans_fam = MiniBatchKMeans(n_clusters=n_classes_fam, random_state=SEED, batch_size=1024)
        cluster_fam = kmeans_fam.fit_predict(E_test_fam_eval)

        sil_fam = float(silhouette_score(
            E_test_fam_eval, y_test_fam_eval,
            sample_size=min(4000, len(E_test_fam_eval)),
            random_state=SEED
        ))
        db_fam = float(davies_bouldin_score(E_test_fam_eval, y_test_fam_eval))
        purity_fam = float(cluster_purity_score(y_test_fam_eval, cluster_fam))

        probe_fam = LogisticRegression(
            solver="lbfgs",
            max_iter=400,
            multi_class="auto",
            random_state=SEED
        )
        probe_fam.fit(E_train_fam, y_train_fam_enc)
        pred_fam = probe_fam.predict(E_test_fam)
        probe_fam_f1 = float(f1_score(y_test_fam_enc, pred_fam, average="macro", zero_division=0))

        table_rows.append({
            "label_space": "attack_family",
            "num_classes": int(n_classes_fam),
            "num_samples_eval": int(len(E_test_fam_eval)),
            "silhouette_score": sil_fam,
            "davies_bouldin_index": db_fam,
            "cluster_purity": purity_fam,
            "linear_probe_macro_f1": probe_fam_f1
        })

table5 = pd.DataFrame(table_rows)
table5_path = os.path.join(OUT_DIR, "table5_embedding_quality_and_linear_probe_results.csv")
table5.to_csv(table5_path, index=False)

overall_bar.update(1)

# =========================================================
# Figure 5 projection
# =========================================================
projector, projector_name, proj_sample_n = try_make_projector()

proj_idx = safe_sample_indices(len(E_test), proj_sample_n, seed=SEED + 10)
E_proj = E_test[proj_idx]
y_proj_bin = y_test[proj_idx]
fam_proj = fam_test.iloc[proj_idx].astype(str).reset_index(drop=True)

if projector_name == "t-SNE":
    pca_pre = PCA(n_components=min(20, E_proj.shape[1]), random_state=SEED)
    E_proj_input = pca_pre.fit_transform(E_proj)
else:
    E_proj_input = E_proj

proj_bar = tqdm(total=1, desc=f"{projector_name} projection", position=1, leave=False)
coords = projector.fit_transform(E_proj_input)
proj_bar.update(1)
proj_bar.close()

proj_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "binary_label": np.where(y_proj_bin == 1, "malicious", "benign"),
    "attack_family": fam_proj
})

top_families = (
    proj_df.loc[proj_df["attack_family"] != "benign", "attack_family"]
    .value_counts()
    .head(TOP_FAMILIES_FOR_FIGURE)
    .index.tolist()
)
proj_df["attack_family_plot"] = proj_df["attack_family"].where(
    proj_df["attack_family"].isin(top_families + ["benign"]),
    "other"
)

proj_df.to_csv(os.path.join(OUT_DIR, "figure5_projection_sample_points.csv"), index=False)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for lab in ["benign", "malicious"]:
    part = proj_df[proj_df["binary_label"] == lab]
    axes[0].scatter(part["x"], part["y"], s=8, alpha=0.55, label=lab)
axes[0].set_title(f"{projector_name} projection by binary class")
axes[0].set_xlabel("Component 1")
axes[0].set_ylabel("Component 2")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.20)

right_order = ["benign"] + [f for f in top_families]
if "other" in proj_df["attack_family_plot"].values:
    right_order += ["other"]

for lab in right_order:
    part = proj_df[proj_df["attack_family_plot"] == lab]
    if len(part) == 0:
        continue
    axes[1].scatter(part["x"], part["y"], s=8, alpha=0.55, label=lab)

axes[1].set_title(f"{projector_name} projection by attack family")
axes[1].set_xlabel("Component 1")
axes[1].set_ylabel("Component 2")
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].grid(alpha=0.20)

fig.suptitle("Figure 5. Two-Dimensional Projection of Learned Embeddings", y=1.02, fontsize=14)
fig.tight_layout()

fig_png = os.path.join(OUT_DIR, "figure5_two_dimensional_projection_of_learned_embeddings.png")
fig_pdf = os.path.join(OUT_DIR, "figure5_two_dimensional_projection_of_learned_embeddings.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

overall_bar.update(1)
overall_bar.close()



In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, confusion_matrix

warnings.filterwarnings("ignore")

# =========================================================
# Paths
# =========================================================
DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
RESULTS4_DIR = "/kaggle/working/results/4_Ablation Study of Model Components"
OUT_DIR = "/kaggle/working/results/6_Calibration and Conformal Analysis"

os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# Speed controls
# =========================================================
TRAIN_CAP = 300000
VAL_CAP = 120000
TEST_CAP = 150000
SEED = 42

# Split-conformal target coverage
CONFORMAL_ALPHA = 0.10   # target coverage ~ 90%

# Reliability diagram bins
N_BINS = 15

# =========================================================
# Helpers
# =========================================================
def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(np.int8)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def build_feature_groups(feature_cols):
    temporal_tokens = [
        "time", "timestamp", "delta", "interval", "jitter", "window", "seq", "age",
        "speed", "heading", "acc", "accel", "velocity", "position", "pos", "x", "y", "z",
        "lat", "lon", "mean", "std", "var", "roll", "rate", "trend"
    ]
    graph_tokens = [
        "neighbor", "neigh", "graph", "degree", "adj", "edge", "hop", "cluster",
        "density", "local", "disagree", "context", "spatial", "distance", "dist",
        "relative", "consensus", "community"
    ]

    temporal = []
    graph = []

    for f in feature_cols:
        fl = f.lower()
        is_temporal = any(tok in fl for tok in temporal_tokens)
        is_graph = any(tok in fl for tok in graph_tokens)

        if is_temporal and not is_graph:
            temporal.append(f)
        elif is_graph and not is_temporal:
            graph.append(f)

    remaining = [f for f in feature_cols if f not in temporal and f not in graph]

    target_min = min(2, len(feature_cols))
    while len(temporal) < target_min and remaining:
        temporal.append(remaining.pop(0))
    while len(graph) < target_min and remaining:
        graph.append(remaining.pop(0))

    if len(temporal) == 0:
        temporal = feature_cols[:min(2, len(feature_cols))]
    if len(graph) == 0:
        leftover = [f for f in feature_cols if f not in temporal]
        graph = leftover[:min(2, len(leftover))] if leftover else temporal[:]

    temporal = list(dict.fromkeys(temporal))
    graph = list(dict.fromkeys(graph))

    if len(temporal) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in temporal:
                temporal.append(f)
            if len(temporal) >= 2:
                break

    if len(graph) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in graph:
                graph.append(f)
            if len(graph) >= 2:
                break

    rows = []
    for f in feature_cols:
        if f in temporal and f in graph:
            grp = "both"
        elif f in temporal:
            grp = "temporal"
        elif f in graph:
            grp = "graph"
        else:
            grp = "shared_other"
        rows.append({"feature": f, "group": grp})

    return temporal, graph, pd.DataFrame(rows)

def fit_safe_latent_projection(X_train, X_val, X_test, max_comp=6, random_state=42, branch_name="branch"):
    n_features = X_train.shape[1]
    if n_features == 0:
        raise ValueError(f"{branch_name} has zero features")

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    if n_features == 1:
        info = {
            "branch": branch_name,
            "method": "identity_1d",
            "input_dim": int(n_features),
            "output_dim": 1
        }
        return X_train_s, X_val_s, X_test_s, info

    n_comp = max(1, min(max_comp, n_features - 1))
    reducer = TruncatedSVD(n_components=n_comp, random_state=random_state)

    Z_train = reducer.fit_transform(X_train_s)
    Z_val = reducer.transform(X_val_s)
    Z_test = reducer.transform(X_test_s)

    info = {
        "branch": branch_name,
        "method": "truncated_svd",
        "input_dim": int(n_features),
        "output_dim": int(n_comp)
    }
    return Z_train, Z_val, Z_test, info

def ece_score_binary(y_true, prob_pos, n_bins=15):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)
    conf = np.maximum(prob_pos, 1.0 - prob_pos)
    pred = (prob_pos >= 0.5).astype(np.int8)
    correct = (pred == y_true).astype(np.float32)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == 0:
            mask = (conf >= bins[i]) & (conf <= bins[i + 1])
        else:
            mask = (conf > bins[i]) & (conf <= bins[i + 1])
        if mask.sum() == 0:
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc_bin - conf_bin)
    return float(ece)

def calibration_curve_points(y_true, prob_pos, n_bins=15):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)
    conf = np.maximum(prob_pos, 1.0 - prob_pos)
    pred = (prob_pos >= 0.5).astype(np.int8)
    correct = (pred == y_true).astype(np.float32)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        if i == 0:
            mask = (conf >= bins[i]) & (conf <= bins[i + 1])
        else:
            mask = (conf > bins[i]) & (conf <= bins[i + 1])

        if mask.sum() == 0:
            continue

        rows.append({
            "bin_id": i,
            "bin_left": float(bins[i]),
            "bin_right": float(bins[i + 1]),
            "bin_center": float((bins[i] + bins[i + 1]) / 2.0),
            "avg_confidence": float(conf[mask].mean()),
            "avg_accuracy": float(correct[mask].mean()),
            "count": int(mask.sum())
        })
    return pd.DataFrame(rows)

def false_alarm_rate(y_true, prob_pos, threshold=0.5):
    y_true = np.asarray(y_true).astype(np.int8)
    y_pred = (np.asarray(prob_pos) >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0

def split_conformal_threshold(y_true_cal, prob_pos_cal, alpha=0.10):
    y_true_cal = np.asarray(y_true_cal).astype(np.int8)
    prob_pos_cal = np.asarray(prob_pos_cal)

    prob_true = np.where(y_true_cal == 1, prob_pos_cal, 1.0 - prob_pos_cal)
    scores = 1.0 - prob_true
    n = len(scores)

    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(1.0, q_level)
    qhat = float(np.quantile(scores, q_level, method="higher"))
    return qhat

def empirical_conformal_coverage(y_true, prob_pos, qhat):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)

    prob_true = np.where(y_true == 1, prob_pos, 1.0 - prob_pos)
    covered = (1.0 - prob_true) <= qhat
    return float(covered.mean())

def risk_coverage_curve(y_true, prob_pos, n_points=25):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)

    conf = np.maximum(prob_pos, 1.0 - prob_pos)
    pred = (prob_pos >= 0.5).astype(np.int8)
    correct = (pred == y_true).astype(np.float32)
    risk = 1.0 - correct

    order = np.argsort(-conf)
    risk_sorted = risk[order]

    rows = []
    n = len(y_true)
    for frac in np.linspace(0.05, 1.0, n_points):
        k = max(1, int(round(frac * n)))
        rows.append({
            "coverage": float(k / n),
            "risk": float(risk_sorted[:k].mean())
        })
    return pd.DataFrame(rows)

# =========================================================
# Load metadata
# =========================================================
with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

feature_group_path = os.path.join(RESULTS4_DIR, "feature_group_assignments.csv")
if os.path.exists(feature_group_path):
    fg = pd.read_csv(feature_group_path)
    temporal_cols = fg.loc[fg["group"].isin(["temporal", "both"]), "feature"].tolist()
    graph_cols = fg.loc[fg["group"].isin(["graph", "both"]), "feature"].tolist()
    if len(temporal_cols) < 2 or len(graph_cols) < 1:
        temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)
    else:
        feature_group_df = fg.copy()
else:
    temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)

# =========================================================
# Load data
# =========================================================
overall_bar = tqdm(total=12, desc="overall", position=0)

needed_cols = set(feature_cols + [label_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)

# =========================================================
# Split + target
# =========================================================
df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

df["target"] = y.astype(np.int8)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
overall_bar.update(1)

# =========================================================
# Numeric features
# =========================================================
X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")
overall_bar.update(1)

# =========================================================
# Split and cap
# =========================================================
train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

y_train_full = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
y_val_full = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
y_test_full = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)

train_idx_local = stratified_cap_indices(y_train_full, TRAIN_CAP, seed=SEED)
val_idx_local = stratified_cap_indices(y_val_full, VAL_CAP, seed=SEED + 1)
test_idx_local = stratified_cap_indices(y_test_full, TEST_CAP, seed=SEED + 2)

X_train = X.loc[train_mask].iloc[train_idx_local].reset_index(drop=True)
y_train = y_train_full[train_idx_local]

X_val = X.loc[val_mask].iloc[val_idx_local].reset_index(drop=True)
y_val = y_val_full[val_idx_local]

X_test = X.loc[test_mask].iloc[test_idx_local].reset_index(drop=True)
y_test = y_test_full[test_idx_local]
overall_bar.update(1)

# =========================================================
# Impute
# =========================================================
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)
overall_bar.update(1)

# =========================================================
# Branch inputs
# =========================================================
feat_to_idx = {f: i for i, f in enumerate(feature_cols)}
temporal_idx = [feat_to_idx[f] for f in temporal_cols if f in feat_to_idx]
graph_idx = [feat_to_idx[f] for f in graph_cols if f in feat_to_idx]

if len(temporal_idx) == 0:
    temporal_idx = list(range(min(2, len(feature_cols))))
if len(graph_idx) == 0:
    graph_idx = list(range(min(2, len(feature_cols))))

X_train_temporal = X_train_imp[:, temporal_idx]
X_val_temporal = X_val_imp[:, temporal_idx]
X_test_temporal = X_test_imp[:, temporal_idx]

X_train_graph = X_train_imp[:, graph_idx]
X_val_graph = X_val_imp[:, graph_idx]
X_test_graph = X_test_imp[:, graph_idx]
overall_bar.update(1)

# =========================================================
# Safe latent branches
# =========================================================
latent_bar = tqdm(total=3, desc="latent branches", position=1, leave=False)

Z_train_temp, Z_val_temp, Z_test_temp, info_temp = fit_safe_latent_projection(
    X_train_temporal, X_val_temporal, X_test_temporal,
    max_comp=6, random_state=SEED, branch_name="temporal"
)
latent_bar.update(1)

Z_train_graph, Z_val_graph, Z_test_graph, info_graph = fit_safe_latent_projection(
    X_train_graph, X_val_graph, X_test_graph,
    max_comp=6, random_state=SEED + 1, branch_name="graph"
)
latent_bar.update(1)

Z_train_ssl, Z_val_ssl, Z_test_ssl, info_ssl = fit_safe_latent_projection(
    X_train_imp, X_val_imp, X_test_imp,
    max_comp=8, random_state=SEED + 2, branch_name="ssl_proxy"
)
latent_bar.update(1)
latent_bar.close()
overall_bar.update(1)

# =========================================================
# Train branches + fusion
# =========================================================
fit_bar = tqdm(total=4, desc="fit models", position=1, leave=False)

model_temp = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED)
model_graph = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 1)
model_ssl = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 2)

model_temp.fit(Z_train_temp, y_train)
fit_bar.update(1)

model_graph.fit(Z_train_graph, y_train)
fit_bar.update(1)

model_ssl.fit(Z_train_ssl, y_train)
fit_bar.update(1)

p_val_temp = model_temp.predict_proba(Z_val_temp)[:, 1]
p_test_temp = model_temp.predict_proba(Z_test_temp)[:, 1]

p_val_graph = model_graph.predict_proba(Z_val_graph)[:, 1]
p_test_graph = model_graph.predict_proba(Z_test_graph)[:, 1]

p_val_ssl = model_ssl.predict_proba(Z_val_ssl)[:, 1]
p_test_ssl = model_ssl.predict_proba(Z_test_ssl)[:, 1]

fusion = LogisticRegression(solver="lbfgs", max_iter=200, random_state=SEED)
fusion.fit(
    np.column_stack([p_val_temp, p_val_graph, p_val_ssl]),
    y_val
)
fit_bar.update(1)
fit_bar.close()
overall_bar.update(1)

p_val_raw = fusion.predict_proba(np.column_stack([p_val_temp, p_val_graph, p_val_ssl]))[:, 1]
p_test_raw = fusion.predict_proba(np.column_stack([p_test_temp, p_test_graph, p_test_ssl]))[:, 1]

# =========================================================
# Split validation into calibration and threshold/conformal parts
# =========================================================
val_idx = np.arange(len(y_val))
rs = np.random.RandomState(SEED)
rs.shuffle(val_idx)

split1 = int(0.50 * len(val_idx))
cal_idx = val_idx[:split1]
conf_idx = val_idx[split1:]

y_val_cal = y_val[cal_idx]
y_val_conf = y_val[conf_idx]

p_val_raw_cal = p_val_raw[cal_idx]
p_val_raw_conf = p_val_raw[conf_idx]
overall_bar.update(1)

# =========================================================
# Isotonic calibration
# =========================================================
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_val_raw_cal, y_val_cal)

p_val_calibrated_conf = iso.transform(p_val_raw_conf)
p_test_calibrated = iso.transform(p_test_raw)
overall_bar.update(1)

# =========================================================
# Reliability, Brier, FAR, Conformal, Risk-Coverage
# =========================================================
curve_before = calibration_curve_points(y_test, p_test_raw, n_bins=N_BINS)
curve_after = calibration_curve_points(y_test, p_test_calibrated, n_bins=N_BINS)

curve_before.to_csv(os.path.join(OUT_DIR, "reliability_before_calibration.csv"), index=False)
curve_after.to_csv(os.path.join(OUT_DIR, "reliability_after_calibration.csv"), index=False)

ece_before = ece_score_binary(y_test, p_test_raw, n_bins=N_BINS)
ece_after = ece_score_binary(y_test, p_test_calibrated, n_bins=N_BINS)

brier_before = float(brier_score_loss(y_test, p_test_raw))
brier_after = float(brier_score_loss(y_test, p_test_calibrated))

far_before = false_alarm_rate(y_test, p_test_raw, threshold=0.5)
far_after = false_alarm_rate(y_test, p_test_calibrated, threshold=0.5)

qhat = split_conformal_threshold(y_val_conf, p_val_calibrated_conf, alpha=CONFORMAL_ALPHA)
emp_cov = empirical_conformal_coverage(y_test, p_test_calibrated, qhat)

risk_cov_before = risk_coverage_curve(y_test, p_test_raw, n_points=25)
risk_cov_after = risk_coverage_curve(y_test, p_test_calibrated, n_points=25)
risk_cov_before["stage"] = "before_calibration"
risk_cov_after["stage"] = "after_calibration"
risk_cov = pd.concat([risk_cov_before, risk_cov_after], ignore_index=True)
risk_cov.to_csv(os.path.join(OUT_DIR, "risk_coverage_curve.csv"), index=False)
overall_bar.update(1)

# =========================================================
# Table 6
# =========================================================
table6 = pd.DataFrame([
    {
        "stage": "before_calibration",
        "ece": ece_before,
        "brier_score": brier_before,
        "conformal_threshold": qhat,
        "empirical_coverage": emp_cov,
        "post_calibration_false_alarm_rate": far_before
    },
    {
        "stage": "after_calibration",
        "ece": ece_after,
        "brier_score": brier_after,
        "conformal_threshold": qhat,
        "empirical_coverage": emp_cov,
        "post_calibration_false_alarm_rate": far_after
    }
])

table6_path = os.path.join(OUT_DIR, "table6_calibration_and_conformal_detection_summary.csv")
table6.to_csv(table6_path, index=False)
overall_bar.update(1)

# =========================================================
# Figure 6
# =========================================================
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

# Reliability before
axes[0].plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Ideal")
axes[0].plot(curve_before["avg_confidence"], curve_before["avg_accuracy"], marker="o", linewidth=2, label="Before")
axes[0].set_title("Reliability diagram before calibration")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Accuracy")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

# Reliability after
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Ideal")
axes[1].plot(curve_after["avg_confidence"], curve_after["avg_accuracy"], marker="o", linewidth=2, label="After")
axes[1].set_title("Reliability diagram after calibration")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Accuracy")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

# Risk-coverage
axes[2].plot(risk_cov_before["coverage"], risk_cov_before["risk"], linewidth=2, label="Before")
axes[2].plot(risk_cov_after["coverage"], risk_cov_after["risk"], linewidth=2, label="After")
axes[2].set_title("Risk-coverage curve")
axes[2].set_xlabel("Coverage")
axes[2].set_ylabel("Risk")
axes[2].set_xlim(0, 1.0)
axes[2].set_ylim(bottom=0)
axes[2].grid(alpha=0.25)
axes[2].legend(frameon=False)

fig.suptitle("Figure 6. Reliability Diagram and Risk-Coverage Curve", y=1.03, fontsize=14)
fig.tight_layout()

fig_png = os.path.join(OUT_DIR, "figure6_reliability_diagram_and_risk_coverage_curve.png")
fig_pdf = os.path.join(OUT_DIR, "figure6_reliability_diagram_and_risk_coverage_curve.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

overall_bar.update(1)
overall_bar.close()


In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    recall_score
)

warnings.filterwarnings("ignore")

# =========================================================
# Paths
# =========================================================
DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"
BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
RESULTS4_DIR = "/kaggle/working/results/4_Ablation Study of Model Components"
OUT_DIR = "/kaggle/working/results/7_Cross Scenario and Out of Distribution Performance"

os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# Speed controls
# =========================================================
TRAIN_FIT_CAP = 300000
ID_EVAL_CAP = 120000
VAL_CAP = 120000
OOD_TEST_CAP = 150000
SEED = 42

# =========================================================
# Helpers
# =========================================================
def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(np.int8)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def build_feature_groups(feature_cols):
    temporal_tokens = [
        "time", "timestamp", "delta", "interval", "jitter", "window", "seq", "age",
        "speed", "heading", "acc", "accel", "velocity", "position", "pos", "x", "y", "z",
        "lat", "lon", "mean", "std", "var", "roll", "rate", "trend"
    ]
    graph_tokens = [
        "neighbor", "neigh", "graph", "degree", "adj", "edge", "hop", "cluster",
        "density", "local", "disagree", "context", "spatial", "distance", "dist",
        "relative", "consensus", "community"
    ]

    temporal = []
    graph = []

    for f in feature_cols:
        fl = f.lower()
        is_temporal = any(tok in fl for tok in temporal_tokens)
        is_graph = any(tok in fl for tok in graph_tokens)

        if is_temporal and not is_graph:
            temporal.append(f)
        elif is_graph and not is_temporal:
            graph.append(f)

    remaining = [f for f in feature_cols if f not in temporal and f not in graph]

    target_min = min(2, len(feature_cols))
    while len(temporal) < target_min and remaining:
        temporal.append(remaining.pop(0))
    while len(graph) < target_min and remaining:
        graph.append(remaining.pop(0))

    if len(temporal) == 0:
        temporal = feature_cols[:min(2, len(feature_cols))]
    if len(graph) == 0:
        leftover = [f for f in feature_cols if f not in temporal]
        graph = leftover[:min(2, len(leftover))] if leftover else temporal[:]

    temporal = list(dict.fromkeys(temporal))
    graph = list(dict.fromkeys(graph))

    if len(temporal) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in temporal:
                temporal.append(f)
            if len(temporal) >= 2:
                break

    if len(graph) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in graph:
                graph.append(f)
            if len(graph) >= 2:
                break

    rows = []
    for f in feature_cols:
        if f in temporal and f in graph:
            grp = "both"
        elif f in temporal:
            grp = "temporal"
        elif f in graph:
            grp = "graph"
        else:
            grp = "shared_other"
        rows.append({"feature": f, "group": grp})

    return temporal, graph, pd.DataFrame(rows)

def fit_safe_latent_transformer(X_fit, max_comp=6, random_state=42, branch_name="branch"):
    n_features = X_fit.shape[1]
    if n_features == 0:
        raise ValueError(f"{branch_name} has zero features")

    scaler = StandardScaler()
    X_fit_s = scaler.fit_transform(X_fit)

    if n_features == 1:
        def transform_fn(X):
            return scaler.transform(X)
        Z_fit = X_fit_s
        info = {
            "branch": branch_name,
            "method": "identity_1d",
            "input_dim": int(n_features),
            "output_dim": 1
        }
        return Z_fit, transform_fn, info

    n_comp = max(1, min(max_comp, n_features - 1))
    reducer = TruncatedSVD(n_components=n_comp, random_state=random_state)
    Z_fit = reducer.fit_transform(X_fit_s)

    def transform_fn(X):
        return reducer.transform(scaler.transform(X))

    info = {
        "branch": branch_name,
        "method": "truncated_svd",
        "input_dim": int(n_features),
        "output_dim": int(n_comp)
    }
    return Z_fit, transform_fn, info

def macro_auprc_binary(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(np.int8)
    prob_pos = np.asarray(prob_pos)
    prob_2 = np.column_stack([1.0 - prob_pos, prob_pos])
    y_onehot = np.zeros((len(y_true), 2), dtype=np.int8)
    y_onehot[np.arange(len(y_true)), y_true] = 1
    return float(average_precision_score(y_onehot, prob_2, average="macro"))

def choose_best_threshold_macro_f1(y_true, prob_pos):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_thr = 0.50
    best_score = -1.0
    for thr in thresholds:
        pred = (prob_pos >= thr).astype(np.int8)
        score = f1_score(y_true, pred, average="macro", zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr

def evaluate_setting(y_true, prob_pos, threshold):
    pred = (np.asarray(prob_pos) >= threshold).astype(np.int8)
    return {
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "macro_auprc": float(macro_auprc_binary(y_true, prob_pos)),
        "recall": float(recall_score(y_true, pred, zero_division=0))
    }

def pct_drop(ref, val):
    denom = ref if abs(ref) > 1e-12 else 1e-12
    return 100.0 * (ref - val) / denom

# =========================================================
# Load metadata
# =========================================================
with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

feature_group_path = os.path.join(RESULTS4_DIR, "feature_group_assignments.csv")
if os.path.exists(feature_group_path):
    fg = pd.read_csv(feature_group_path)
    temporal_cols = fg.loc[fg["group"].isin(["temporal", "both"]), "feature"].tolist()
    graph_cols = fg.loc[fg["group"].isin(["graph", "both"]), "feature"].tolist()
    if len(temporal_cols) < 2 or len(graph_cols) < 1:
        temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)
    else:
        feature_group_df = fg.copy()
else:
    temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)

# =========================================================
# Load data
# =========================================================
overall_bar = tqdm(total=11, desc="overall", position=0)

needed_cols = set(feature_cols + [label_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)

# =========================================================
# Build scenario split and target
# =========================================================
df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

df["target"] = y.astype(np.int8)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
overall_bar.update(1)

# =========================================================
# Numeric features
# =========================================================
X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")
overall_bar.update(1)

# =========================================================
# Split into train/val/test scenario blocks
# =========================================================
train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

X_train_all = X.loc[train_mask].reset_index(drop=True)
y_train_all = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
sid_train_all = df.loc[train_mask, "scenario_id"].astype(str).reset_index(drop=True)

X_val_all = X.loc[val_mask].reset_index(drop=True)
y_val_all = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
sid_val_all = df.loc[val_mask, "scenario_id"].astype(str).reset_index(drop=True)

X_test_all = X.loc[test_mask].reset_index(drop=True)
y_test_all = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)
sid_test_all = df.loc[test_mask, "scenario_id"].astype(str).reset_index(drop=True)
overall_bar.update(1)

# =========================================================
# ID holdout from seen training scenarios
# OOD from held-out test scenarios
# =========================================================
train_fit_idx, id_eval_idx = train_test_split(
    np.arange(len(y_train_all)),
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_all
)

train_fit_cap = stratified_cap_indices(y_train_all[train_fit_idx], TRAIN_FIT_CAP, seed=SEED)
id_eval_cap = stratified_cap_indices(y_train_all[id_eval_idx], ID_EVAL_CAP, seed=SEED + 1)
val_cap = stratified_cap_indices(y_val_all, VAL_CAP, seed=SEED + 2)
ood_cap = stratified_cap_indices(y_test_all, OOD_TEST_CAP, seed=SEED + 3)

train_fit_idx = train_fit_idx[train_fit_cap]
id_eval_idx = id_eval_idx[id_eval_cap]
val_idx = val_cap
ood_idx = ood_cap

X_train_fit = X_train_all.iloc[train_fit_idx].reset_index(drop=True)
y_train_fit = y_train_all[train_fit_idx]
sid_train_fit = sid_train_all.iloc[train_fit_idx].reset_index(drop=True)

X_id = X_train_all.iloc[id_eval_idx].reset_index(drop=True)
y_id = y_train_all[id_eval_idx]
sid_id = sid_train_all.iloc[id_eval_idx].reset_index(drop=True)

X_val = X_val_all.iloc[val_idx].reset_index(drop=True)
y_val = y_val_all[val_idx]
sid_val = sid_val_all.iloc[val_idx].reset_index(drop=True)

X_ood = X_test_all.iloc[ood_idx].reset_index(drop=True)
y_ood = y_test_all[ood_idx]
sid_ood = sid_test_all.iloc[ood_idx].reset_index(drop=True)
overall_bar.update(1)

# =========================================================
# Impute with training-fit only
# =========================================================
imputer = SimpleImputer(strategy="median")
X_train_fit_imp = imputer.fit_transform(X_train_fit)
X_id_imp = imputer.transform(X_id)
X_val_imp = imputer.transform(X_val)
X_ood_imp = imputer.transform(X_ood)
overall_bar.update(1)

# =========================================================
# Branch feature groups
# =========================================================
feat_to_idx = {f: i for i, f in enumerate(feature_cols)}
temporal_idx = [feat_to_idx[f] for f in temporal_cols if f in feat_to_idx]
graph_idx = [feat_to_idx[f] for f in graph_cols if f in feat_to_idx]

if len(temporal_idx) == 0:
    temporal_idx = list(range(min(2, len(feature_cols))))
if len(graph_idx) == 0:
    graph_idx = list(range(min(2, len(feature_cols))))

X_train_fit_temporal = X_train_fit_imp[:, temporal_idx]
X_id_temporal = X_id_imp[:, temporal_idx]
X_val_temporal = X_val_imp[:, temporal_idx]
X_ood_temporal = X_ood_imp[:, temporal_idx]

X_train_fit_graph = X_train_fit_imp[:, graph_idx]
X_id_graph = X_id_imp[:, graph_idx]
X_val_graph = X_val_imp[:, graph_idx]
X_ood_graph = X_ood_imp[:, graph_idx]
overall_bar.update(1)

# =========================================================
# Latent branches
# =========================================================
latent_bar = tqdm(total=3, desc="latent branches", position=1, leave=False)

Z_train_temp, temp_transform, info_temp = fit_safe_latent_transformer(
    X_train_fit_temporal, max_comp=6, random_state=SEED, branch_name="temporal"
)
Z_id_temp = temp_transform(X_id_temporal)
Z_val_temp = temp_transform(X_val_temporal)
Z_ood_temp = temp_transform(X_ood_temporal)
latent_bar.update(1)

Z_train_graph, graph_transform, info_graph = fit_safe_latent_transformer(
    X_train_fit_graph, max_comp=6, random_state=SEED + 1, branch_name="graph"
)
Z_id_graph = graph_transform(X_id_graph)
Z_val_graph = graph_transform(X_val_graph)
Z_ood_graph = graph_transform(X_ood_graph)
latent_bar.update(1)

Z_train_ssl, ssl_transform, info_ssl = fit_safe_latent_transformer(
    X_train_fit_imp, max_comp=8, random_state=SEED + 2, branch_name="ssl_proxy"
)
Z_id_ssl = ssl_transform(X_id_imp)
Z_val_ssl = ssl_transform(X_val_imp)
Z_ood_ssl = ssl_transform(X_ood_imp)
latent_bar.update(1)
latent_bar.close()
overall_bar.update(1)

# =========================================================
# Train branches and fusion
# =========================================================
fit_bar = tqdm(total=4, desc="fit models", position=1, leave=False)

model_temp = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED)
model_graph = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 1)
model_ssl = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 2)

model_temp.fit(Z_train_temp, y_train_fit)
fit_bar.update(1)

model_graph.fit(Z_train_graph, y_train_fit)
fit_bar.update(1)

model_ssl.fit(Z_train_ssl, y_train_fit)
fit_bar.update(1)

p_val_temp = model_temp.predict_proba(Z_val_temp)[:, 1]
p_id_temp = model_temp.predict_proba(Z_id_temp)[:, 1]
p_ood_temp = model_temp.predict_proba(Z_ood_temp)[:, 1]

p_val_graph = model_graph.predict_proba(Z_val_graph)[:, 1]
p_id_graph = model_graph.predict_proba(Z_id_graph)[:, 1]
p_ood_graph = model_graph.predict_proba(Z_ood_graph)[:, 1]

p_val_ssl = model_ssl.predict_proba(Z_val_ssl)[:, 1]
p_id_ssl = model_ssl.predict_proba(Z_id_ssl)[:, 1]
p_ood_ssl = model_ssl.predict_proba(Z_ood_ssl)[:, 1]

# stacker trained on validation scenarios
val_fuse_idx, val_rest_idx = train_test_split(
    np.arange(len(y_val)),
    test_size=0.60,
    random_state=SEED,
    stratify=y_val
)
val_cal_idx, val_thr_idx = train_test_split(
    val_rest_idx,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=y_val[val_rest_idx]
)

fusion = LogisticRegression(solver="lbfgs", max_iter=200, random_state=SEED)
fusion.fit(
    np.column_stack([p_val_temp[val_fuse_idx], p_val_graph[val_fuse_idx], p_val_ssl[val_fuse_idx]]),
    y_val[val_fuse_idx]
)
fit_bar.update(1)
fit_bar.close()
overall_bar.update(1)

# fused raw scores
p_val_raw = fusion.predict_proba(np.column_stack([p_val_temp, p_val_graph, p_val_ssl]))[:, 1]
p_id_raw = fusion.predict_proba(np.column_stack([p_id_temp, p_id_graph, p_id_ssl]))[:, 1]
p_ood_raw = fusion.predict_proba(np.column_stack([p_ood_temp, p_ood_graph, p_ood_ssl]))[:, 1]

# isotonic calibration on held-out validation subset
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_val_raw[val_cal_idx], y_val[val_cal_idx])

p_val_thr = iso.transform(p_val_raw[val_thr_idx])
p_id = iso.transform(p_id_raw)
p_ood = iso.transform(p_ood_raw)

threshold = choose_best_threshold_macro_f1(y_val[val_thr_idx], p_val_thr)
overall_bar.update(1)

# =========================================================
# Evaluate ID vs OOD
# =========================================================
id_metrics = evaluate_setting(y_id, p_id, threshold)
ood_metrics = evaluate_setting(y_ood, p_ood, threshold)

table7 = pd.DataFrame([
    {
        "evaluation_setting": "in_distribution_seen_scenarios",
        "n_scenarios": int(sid_id.nunique()),
        "n_samples": int(len(y_id)),
        "macro_f1": id_metrics["macro_f1"],
        "macro_auprc": id_metrics["macro_auprc"],
        "recall": id_metrics["recall"],
        "macro_f1_degradation_pct": 0.0,
        "macro_auprc_degradation_pct": 0.0,
        "recall_degradation_pct": 0.0
    },
    {
        "evaluation_setting": "out_of_distribution_held_out_scenarios",
        "n_scenarios": int(sid_ood.nunique()),
        "n_samples": int(len(y_ood)),
        "macro_f1": ood_metrics["macro_f1"],
        "macro_auprc": ood_metrics["macro_auprc"],
        "recall": ood_metrics["recall"],
        "macro_f1_degradation_pct": pct_drop(id_metrics["macro_f1"], ood_metrics["macro_f1"]),
        "macro_auprc_degradation_pct": pct_drop(id_metrics["macro_auprc"], ood_metrics["macro_auprc"]),
        "recall_degradation_pct": pct_drop(id_metrics["recall"], ood_metrics["recall"])
    }
])

table7_path = os.path.join(OUT_DIR, "table7_robustness_under_scenario_shift.csv")
table7.to_csv(table7_path, index=False)

split_summary = pd.DataFrame([
    {"subset": "train_fit_seen_scenarios", "n_samples": len(y_train_fit), "n_scenarios": int(sid_train_fit.nunique()), "positive_rate": float(np.mean(y_train_fit))},
    {"subset": "id_eval_seen_scenarios", "n_samples": len(y_id), "n_scenarios": int(sid_id.nunique()), "positive_rate": float(np.mean(y_id))},
    {"subset": "validation_heldout_scenarios", "n_samples": len(y_val), "n_scenarios": int(sid_val.nunique()), "positive_rate": float(np.mean(y_val))},
    {"subset": "ood_test_heldout_scenarios", "n_samples": len(y_ood), "n_scenarios": int(sid_ood.nunique()), "positive_rate": float(np.mean(y_ood))}
])
split_summary.to_csv(os.path.join(OUT_DIR, "scenario_shift_split_summary.csv"), index=False)
overall_bar.update(1)

# =========================================================
# Figure 7
# Left: grouped bars on main metrics
# Right: degradation bars
# =========================================================
metric_names = ["Macro F1", "Macro AUPRC", "Recall"]
id_vals = [id_metrics["macro_f1"], id_metrics["macro_auprc"], id_metrics["recall"]]
ood_vals = [ood_metrics["macro_f1"], ood_metrics["macro_auprc"], ood_metrics["recall"]]
deg_vals = [
    pct_drop(id_metrics["macro_f1"], ood_metrics["macro_f1"]),
    pct_drop(id_metrics["macro_auprc"], ood_metrics["macro_auprc"]),
    pct_drop(id_metrics["recall"], ood_metrics["recall"])
]

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8))

# grouped bars
x = np.arange(len(metric_names))
width = 0.34
axes[0].bar(x - width/2, id_vals, width=width, label="In-distribution")
axes[0].bar(x + width/2, ood_vals, width=width, label="Held-out scenarios")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].set_ylim(0, 1.02)
axes[0].set_ylabel("Score")
axes[0].set_title("ID versus held-out scenario performance")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.25)

# degradation bars
axes[1].bar(metric_names, deg_vals)
axes[1].set_ylabel("Degradation (%)")
axes[1].set_title("Performance degradation under scenario shift")
axes[1].grid(axis="y", alpha=0.25)
for i, v in enumerate(deg_vals):
    axes[1].text(i, v, f"{v:.2f}%", ha="center", va="bottom")

fig.suptitle("Figure 7. Cross-Scenario and Out-of-Distribution Performance", y=1.03, fontsize=14)
fig.tight_layout()

fig_png = os.path.join(OUT_DIR, "figure7_cross_scenario_and_out_of_distribution_performance.png")
fig_pdf = os.path.join(OUT_DIR, "figure7_cross_scenario_and_out_of_distribution_performance.pdf")
fig.savefig(fig_png, bbox_inches="tight")
fig.savefig(fig_pdf, bbox_inches="tight")
plt.close(fig)

overall_bar.update(1)
overall_bar.close()



In [ ]:
# =========================================================
# FIGURE 8 + TABLE 8 + FIGURE 9 + TABLE 9
# One-cell snippet
# =========================================================
import os
import json
import time
import tracemalloc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

# =========================================================
# Paths
# =========================================================
DATA_PATH = "/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb/mixalldata_clean.csv"

BENCH_DIR = "/kaggle/working/results/1_Benchmark Summary and Evaluation Protocol"
RESULTS2_DIR = "/kaggle/working/results/2_Main Detection Results"
RESULTS3_DIR = "/kaggle/working/results/3_Per Attack Family Analysis"
RESULTS4_DIR = "/kaggle/working/results/4_Ablation Study of Model Components"
RESULTS7_DIR = "/kaggle/working/results/7_Cross Scenario and Out of Distribution Performance"

OUT_DIR = "/kaggle/working/results/8_Runtime Tradeoff and Case Study Analysis"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# Speed controls
# =========================================================
RESOURCE_TRAIN_CAP = 120000
RESOURCE_VAL_CAP = 40000
RESOURCE_TEST_CAP = 60000

FULLMODEL_TRAIN_CAP = 180000
FULLMODEL_VAL_CAP = 80000
FULLMODEL_TEST_CAP = 100000

SEED = 42

# =========================================================
# Helpers
# =========================================================
def infer_binary_label(series):
    s = series.copy()
    num = pd.to_numeric(s, errors="coerce")
    nunq_num = num.nunique(dropna=True)

    if num.notna().mean() >= 0.90 and 1 < nunq_num <= 100:
        uniq = sorted(pd.Series(num.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            return num.fillna(0).astype(np.int8)
        if 0 in uniq:
            return (num.fillna(0) != 0).astype(np.int8)

    txt = s.astype(str).str.strip().str.lower()
    benign = txt.str.contains(
        r"\b(?:benign|normal|genuine|legit|no[_ -]?attack|non[_ -]?attack|baseline|safe|clean)\b",
        regex=True, na=False
    )
    malicious = txt.str.contains(
        r"\b(?:malicious|attack|attacker|spoof|bogus|sybil|dos|fake|ghost|replay|wormhole|jam)\b",
        regex=True, na=False
    )

    y = pd.Series(np.nan, index=s.index, dtype="float32")
    y.loc[benign] = 0
    y.loc[malicious] = 1

    if y.notna().mean() >= 0.05 and y.nunique(dropna=True) >= 2:
        return y.fillna(0).astype(np.int8)

    return None

def normalize_attack_family(series, y_binary=None):
    x = series.astype(str).str.strip().str.lower()
    x = x.str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()

    replace_map = {
        "": "",
        "nan": "",
        "none": "",
        "null": "",
        "false": "",
        "true": "unknown_malicious",
        "0": "",
        "1": "unknown_malicious",
        "benign": "benign",
        "normal": "benign",
        "genuine": "benign",
        "clean": "benign",
        "safe": "benign",
        "baseline": "benign",
        "malicious": "unknown_malicious",
        "attack": "unknown_malicious",
        "attacker": "unknown_malicious",
    }
    x = x.replace(replace_map)

    if y_binary is not None:
        y_binary = pd.Series(y_binary, index=series.index)
        x = x.where(y_binary.astype(int) == 1, "benign")
        x = x.where(~((y_binary.astype(int) == 1) & (x == "")), "unknown_malicious")

    return x.astype(str)

def make_scenario_id(df, selected_columns):
    sender_col = selected_columns.get("sender_col")
    scenario_candidates = selected_columns.get("scenario_candidates", [])
    scenario_candidates = [c for c in scenario_candidates if c in df.columns]

    if len(scenario_candidates) > 0:
        sid = df[scenario_candidates[0]].astype(str).fillna("NA")
        for c in scenario_candidates[1:]:
            sid = sid + "|" + df[c].astype(str).fillna("NA")
        return sid.astype(str)

    if sender_col is not None and sender_col in df.columns:
        return ("sender_fallback|" + df[sender_col].astype(str).fillna("UNK")).astype(str)

    return pd.Series(["global_0"] * len(df), index=df.index, dtype="object")

def stratified_cap_indices(y, max_rows, seed=42):
    y = np.asarray(y).astype(np.int8)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)

    idx_all = np.arange(n)
    pos_idx = idx_all[y == 1]
    neg_idx = idx_all[y == 0]

    rs = np.random.RandomState(seed)
    pos_take = max(1, int(round(max_rows * len(pos_idx) / n)))
    neg_take = max(1, max_rows - pos_take)

    pos_sel = rs.choice(pos_idx, size=min(len(pos_idx), pos_take), replace=False)
    neg_sel = rs.choice(neg_idx, size=min(len(neg_idx), neg_take), replace=False)

    out = np.concatenate([pos_sel, neg_sel])
    rs.shuffle(out)
    return out

def build_feature_groups(feature_cols):
    temporal_tokens = [
        "time", "timestamp", "delta", "interval", "jitter", "window", "seq", "age",
        "speed", "heading", "acc", "accel", "velocity", "position", "pos", "x", "y", "z",
        "lat", "lon", "mean", "std", "var", "roll", "rate", "trend"
    ]
    graph_tokens = [
        "neighbor", "neigh", "graph", "degree", "adj", "edge", "hop", "cluster",
        "density", "local", "disagree", "context", "spatial", "distance", "dist",
        "relative", "consensus", "community"
    ]

    temporal = []
    graph = []

    for f in feature_cols:
        fl = f.lower()
        is_temporal = any(tok in fl for tok in temporal_tokens)
        is_graph = any(tok in fl for tok in graph_tokens)

        if is_temporal and not is_graph:
            temporal.append(f)
        elif is_graph and not is_temporal:
            graph.append(f)

    remaining = [f for f in feature_cols if f not in temporal and f not in graph]
    target_min = min(2, len(feature_cols))

    while len(temporal) < target_min and remaining:
        temporal.append(remaining.pop(0))
    while len(graph) < target_min and remaining:
        graph.append(remaining.pop(0))

    if len(temporal) == 0:
        temporal = feature_cols[:min(2, len(feature_cols))]
    if len(graph) == 0:
        leftover = [f for f in feature_cols if f not in temporal]
        graph = leftover[:min(2, len(leftover))] if leftover else temporal[:]

    temporal = list(dict.fromkeys(temporal))
    graph = list(dict.fromkeys(graph))

    if len(temporal) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in temporal:
                temporal.append(f)
            if len(temporal) >= 2:
                break

    if len(graph) < 2 and len(feature_cols) >= 2:
        for f in feature_cols:
            if f not in graph:
                graph.append(f)
            if len(graph) >= 2:
                break

    rows = []
    for f in feature_cols:
        if f in temporal and f in graph:
            grp = "both"
        elif f in temporal:
            grp = "temporal"
        elif f in graph:
            grp = "graph"
        else:
            grp = "shared_other"
        rows.append({"feature": f, "group": grp})

    return temporal, graph, pd.DataFrame(rows)

def fit_safe_latent_transformer(X_fit, max_comp=6, random_state=42, branch_name="branch"):
    n_features = X_fit.shape[1]
    if n_features == 0:
        raise ValueError(f"{branch_name} has zero features")

    scaler = StandardScaler()
    X_fit_s = scaler.fit_transform(X_fit)

    if n_features == 1:
        def transform_fn(X):
            return scaler.transform(X)
        Z_fit = X_fit_s
        info = {
            "branch": branch_name,
            "method": "identity_1d",
            "input_dim": int(n_features),
            "output_dim": 1
        }
        return Z_fit, transform_fn, info

    n_comp = max(1, min(max_comp, n_features - 1))
    reducer = TruncatedSVD(n_components=n_comp, random_state=random_state)
    Z_fit = reducer.fit_transform(X_fit_s)

    def transform_fn(X):
        return reducer.transform(scaler.transform(X))

    info = {
        "branch": branch_name,
        "method": "truncated_svd",
        "input_dim": int(n_features),
        "output_dim": int(n_comp)
    }
    return Z_fit, transform_fn, info

def choose_best_threshold_macro_f1(y_true, prob_pos):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_thr = 0.50
    best_score = -1.0
    for thr in thresholds:
        pred = (prob_pos >= thr).astype(np.int8)
        score = f1_score(y_true, pred, average="macro", zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr

def model_parameter_count(model):
    try:
        if hasattr(model, "coef_"):
            n = int(np.size(model.coef_))
            if hasattr(model, "intercept_"):
                n += int(np.size(model.intercept_))
            return n

        if hasattr(model, "theta_") and hasattr(model, "var_"):
            n = int(np.size(model.theta_) + np.size(model.var_))
            if hasattr(model, "class_prior_"):
                n += int(np.size(model.class_prior_))
            return n

        if hasattr(model, "class_prior_"):
            return int(np.size(model.class_prior_))

        if hasattr(model, "_predictors"):
            total = 0
            for stage in model._predictors:
                for pred in stage:
                    if hasattr(pred, "nodes"):
                        total += int(pred.nodes.shape[0])
            if total > 0:
                return total
    except Exception:
        pass
    return np.nan

def pareto_frontier_min_x_max_y(df, x_col, y_col):
    pts = df[[x_col, y_col]].to_numpy()
    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        xi, yi = pts[i]
        for j in range(len(df)):
            if i == j:
                continue
            xj, yj = pts[j]
            if (xj <= xi and yj >= yi) and (xj < xi or yj > yi):
                keep[i] = False
                break
    return df.loc[keep].sort_values(x_col)

def compute_feature_weights(X_train_scaled, y_train):
    y = y_train.astype(float)
    y_std = y.std()
    if y_std < 1e-12:
        return np.zeros(X_train_scaled.shape[1], dtype=float)
    yc = (y - y.mean()) / y_std
    Xc = X_train_scaled - X_train_scaled.mean(axis=0, keepdims=True)
    xs = X_train_scaled.std(axis=0)
    xs = np.where(xs < 1e-12, 1.0, xs)
    w = np.abs((Xc * yc[:, None]).mean(axis=0) / xs)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    return w

def local_feature_scores(x_scaled_row, global_weights):
    return np.abs(x_scaled_row) * global_weights

def build_graph_edge_proxy(case_graph_scores, graph_feature_names, graph_corr, top_k_feats=4, top_k_edges=4):
    if len(graph_feature_names) == 0:
        return pd.DataFrame(columns=["edge", "score"])
    if len(graph_feature_names) == 1:
        return pd.DataFrame({
            "edge": [graph_feature_names[0]],
            "score": [float(case_graph_scores[0])]
        })

    feat_idx = np.argsort(-case_graph_scores)[:min(top_k_feats, len(case_graph_scores))]
    rows = []
    for a in range(len(feat_idx)):
        for b in range(a + 1, len(feat_idx)):
            i = feat_idx[a]
            j = feat_idx[b]
            score = abs(graph_corr[i, j]) * case_graph_scores[i] * case_graph_scores[j]
            rows.append({
                "edge": f"{graph_feature_names[i]} ↔ {graph_feature_names[j]}",
                "score": float(score)
            })

    if len(rows) == 0:
        return pd.DataFrame({
            "edge": graph_feature_names[:min(top_k_edges, len(graph_feature_names))],
            "score": case_graph_scores[:min(top_k_edges, len(case_graph_scores))]
        })

    out = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_k_edges)
    return out

def heuristic_reason_and_fix(precision=None, recall=None, failure_type=None):
    if failure_type == "false_positive":
        return (
            "Benign traffic overlaps with attack-like local dynamics or graph-side cues.",
            "Add hard-negative mining, stronger benign diversity, and stricter post-calibration thresholds."
        )
    if failure_type == "false_negative":
        return (
            "Attack signature is weak or temporally diluted under the current decision boundary.",
            "Use family-aware thresholds, recall-focused reweighting, and longer temporal context."
        )
    if precision is not None and recall is not None:
        if recall < 0.30 and precision >= 0.80:
            return (
                "Conservative boundary suppresses subtle positives from this family.",
                "Use family-specific thresholds and recall-oriented training weights."
            )
        if precision < 0.30 and recall >= 0.70:
            return (
                "Benign samples partially mimic this family and trigger over-sensitive alarms.",
                "Use hard-negative mining, better calibration, and feature denoising."
            )
        if precision < 0.50 and recall < 0.50:
            return (
                "Family remains weakly separable in both temporal and graph-side representations.",
                "Increase context length and enrich relational features or explicit graph modeling."
            )
    return (
        "Family signature partially overlaps with neighboring benign or attack patterns.",
        "Improve family-aware supervision, richer context modeling, and targeted threshold selection."
    )

# =========================================================
# Load saved metadata/results
# =========================================================
with open(os.path.join(BENCH_DIR, "benchmark_metadata.json"), "r") as f:
    bench_meta = json.load(f)

with open(os.path.join(RESULTS2_DIR, "run_summary.json"), "r") as f:
    run2_meta = json.load(f)

selected_columns = bench_meta.get("selected_columns", {})
feature_cols = run2_meta["feature_columns"]
label_col = run2_meta["chosen_label_column"]

main_results = pd.read_csv(os.path.join(RESULTS2_DIR, "table2_main_detection_results_across_all_compared_methods.csv"))
table3 = pd.read_csv(os.path.join(RESULTS3_DIR, "table3_class_wise_detection_performance_by_attack_family.csv"))
table3["attack_family"] = table3["attack_family"].astype(str)

family_col = None
run3_path = os.path.join(RESULTS3_DIR, "run_summary.json")
if os.path.exists(run3_path):
    with open(run3_path, "r") as f:
        run3_meta = json.load(f)
    family_col = run3_meta.get("attack_family_column")

if family_col is None:
    fam_diag = pd.read_csv(os.path.join(RESULTS3_DIR, "family_column_diagnostics.csv"))
    family_col = fam_diag.iloc[0]["column"]

feature_group_path = os.path.join(RESULTS4_DIR, "feature_group_assignments.csv")
if os.path.exists(feature_group_path):
    fg = pd.read_csv(feature_group_path)
    temporal_cols = fg.loc[fg["group"].isin(["temporal", "both"]), "feature"].tolist()
    graph_cols = fg.loc[fg["group"].isin(["graph", "both"]), "feature"].tolist()
    if len(temporal_cols) < 2 or len(graph_cols) < 1:
        temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)
    else:
        feature_group_df = fg.copy()
else:
    temporal_cols, graph_cols, feature_group_df = build_feature_groups(feature_cols)

# =========================================================
# Load dataset once
# =========================================================
overall_bar = tqdm(total=11, desc="overall", position=0)

needed_cols = set(feature_cols + [label_col, family_col])
for c in [selected_columns.get("sender_col"), selected_columns.get("time_col")]:
    if c is not None:
        needed_cols.add(c)
for c in selected_columns.get("scenario_candidates", []):
    needed_cols.add(c)

df = pd.read_csv(DATA_PATH, usecols=sorted(needed_cols), low_memory=False)
overall_bar.update(1)

df["scenario_id"] = make_scenario_id(df, selected_columns)

split_manifest_path = os.path.join(BENCH_DIR, "scenario_split_manifest.csv")
if os.path.exists(split_manifest_path):
    manifest = pd.read_csv(split_manifest_path, usecols=["scenario_id", "split"])
    manifest["scenario_id"] = manifest["scenario_id"].astype(str)
    df["scenario_id"] = df["scenario_id"].astype(str)
    df = df.merge(manifest, on="scenario_id", how="left")
else:
    sid_hash = df["scenario_id"].astype(str).map(lambda x: int(str(abs(hash(x))) % 10000))
    df["split"] = np.where(sid_hash < 7000, "train", np.where(sid_hash < 8500, "validation", "test"))

y = infer_binary_label(df[label_col])
if y is None or y.nunique(dropna=True) < 2:
    raise ValueError(f"Could not infer a valid binary label from '{label_col}'")

df["target"] = y.astype(np.int8)
df["attack_family"] = normalize_attack_family(df[family_col], df["target"])
df = df[df["split"].isin(["train", "validation", "test"])].copy()
overall_bar.update(1)

X = pd.DataFrame(index=df.index)
for c in feature_cols:
    X[c] = pd.to_numeric(df[c], errors="coerce")
overall_bar.update(1)

train_mask = df["split"].eq("train").values
val_mask = df["split"].eq("validation").values
test_mask = df["split"].eq("test").values

X_train_all = X.loc[train_mask].reset_index(drop=True)
y_train_all = df.loc[train_mask, "target"].to_numpy(dtype=np.int8)
fam_train_all = df.loc[train_mask, "attack_family"].astype(str).reset_index(drop=True)

X_val_all = X.loc[val_mask].reset_index(drop=True)
y_val_all = df.loc[val_mask, "target"].to_numpy(dtype=np.int8)
fam_val_all = df.loc[val_mask, "attack_family"].astype(str).reset_index(drop=True)

X_test_all = X.loc[test_mask].reset_index(drop=True)
y_test_all = df.loc[test_mask, "target"].to_numpy(dtype=np.int8)
fam_test_all = df.loc[test_mask, "attack_family"].astype(str).reset_index(drop=True)
overall_bar.update(1)

# =========================================================
# PART A: Figure 8 and Table 8
# =========================================================
train_res_idx = stratified_cap_indices(y_train_all, RESOURCE_TRAIN_CAP, seed=SEED)
val_res_idx = stratified_cap_indices(y_val_all, RESOURCE_VAL_CAP, seed=SEED + 1)
test_res_idx = stratified_cap_indices(y_test_all, RESOURCE_TEST_CAP, seed=SEED + 2)

X_train_res = X_train_all.iloc[train_res_idx].reset_index(drop=True)
y_train_res = y_train_all[train_res_idx]
X_val_res = X_val_all.iloc[val_res_idx].reset_index(drop=True)
y_val_res = y_val_all[val_res_idx]
X_test_res = X_test_all.iloc[test_res_idx].reset_index(drop=True)

imputer_res = SimpleImputer(strategy="median")
X_train_res_imp = imputer_res.fit_transform(X_train_res)
X_val_res_imp = imputer_res.transform(X_val_res)
X_test_res_imp = imputer_res.transform(X_test_res)

scaler_res = StandardScaler()
X_train_res_scaled = scaler_res.fit_transform(X_train_res_imp)
X_val_res_scaled = scaler_res.transform(X_val_res_imp)
X_test_res_scaled = scaler_res.transform(X_test_res_imp)
overall_bar.update(1)

compare_models = [
    ("Dummy-Prior", DummyClassifier(strategy="prior", random_state=SEED), "raw"),
    ("GaussianNB", GaussianNB(), "raw"),
    ("SGD-LogLoss", SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=20, tol=1e-3, class_weight="balanced", random_state=SEED), "scaled"),
    ("LogReg-SAGA", LogisticRegression(solver="saga", penalty="l2", C=1.0, max_iter=60, class_weight="balanced", n_jobs=-1, random_state=SEED), "scaled"),
    ("HistGB", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED), "raw"),
]

resource_rows = []
profile_bar = tqdm(compare_models, desc="profile compared methods", position=1, leave=False)

for method, model, mode in profile_bar:
    Xt_train = X_train_res_scaled if mode == "scaled" else X_train_res_imp
    Xt_test = X_test_res_scaled if mode == "scaled" else X_test_res_imp

    tracemalloc.start()
    t0 = time.perf_counter()
    model.fit(Xt_train, y_train_res)
    fit_sec = time.perf_counter() - t0
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    t1 = time.perf_counter()
    proba = model.predict_proba(Xt_test)[:, 1]
    infer_sec = time.perf_counter() - t1

    latency_us = (infer_sec / len(Xt_test)) * 1e6
    throughput = len(Xt_test) / max(infer_sec, 1e-12)

    resource_rows.append({
        "method": method,
        "inference_latency_us_per_sample": float(latency_us),
        "throughput_samples_per_sec": float(throughput),
        "parameter_count": model_parameter_count(model),
        "peak_memory_mb": float(peak_mem / (1024 ** 2)),
        "training_time_s": float(fit_sec)
    })

resource_df = pd.DataFrame(resource_rows)
table8 = resource_df.sort_values("inference_latency_us_per_sample").reset_index(drop=True)
table8_path = os.path.join(OUT_DIR, "table8_runtime_and_resource_consumption_profile.csv")
table8.to_csv(table8_path, index=False)

fig8_df = main_results[["method", "macro_f1"]].merge(
    resource_df[["method", "inference_latency_us_per_sample", "throughput_samples_per_sec"]],
    on="method",
    how="inner"
)

pareto_df = pareto_frontier_min_x_max_y(fig8_df, "inference_latency_us_per_sample", "macro_f1")

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, ax = plt.subplots(figsize=(8.5, 6.2))
ax.scatter(
    fig8_df["inference_latency_us_per_sample"],
    fig8_df["macro_f1"],
    s=90
)
for _, row in fig8_df.iterrows():
    ax.annotate(
        row["method"],
        (row["inference_latency_us_per_sample"], row["macro_f1"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10
    )

if len(pareto_df) >= 2:
    ax.plot(
        pareto_df["inference_latency_us_per_sample"],
        pareto_df["macro_f1"],
        linewidth=1.8,
        linestyle="--",
        label="Pareto frontier"
    )
    ax.legend(frameon=False)

ax.set_xscale("log")
ax.set_xlabel("Inference latency per sample (microseconds, log scale)")
ax.set_ylabel("Macro F1")
ax.set_title("Figure 8. Accuracy-Latency Tradeoff of Compared Methods")
ax.grid(alpha=0.25)

fig.tight_layout()
fig8_png = os.path.join(OUT_DIR, "figure8_accuracy_latency_tradeoff_of_compared_methods.png")
fig8_pdf = os.path.join(OUT_DIR, "figure8_accuracy_latency_tradeoff_of_compared_methods.pdf")
fig.savefig(fig8_png, bbox_inches="tight")
fig.savefig(fig8_pdf, bbox_inches="tight")
plt.close(fig)
overall_bar.update(1)


train_full_idx = stratified_cap_indices(y_train_all, FULLMODEL_TRAIN_CAP, seed=SEED + 10)
val_full_idx = stratified_cap_indices(y_val_all, FULLMODEL_VAL_CAP, seed=SEED + 11)
test_full_idx = stratified_cap_indices(y_test_all, FULLMODEL_TEST_CAP, seed=SEED + 12)

X_train_full = X_train_all.iloc[train_full_idx].reset_index(drop=True)
y_train_full = y_train_all[train_full_idx]
fam_train_full = fam_train_all.iloc[train_full_idx].reset_index(drop=True)

X_val_full = X_val_all.iloc[val_full_idx].reset_index(drop=True)
y_val_full = y_val_all[val_full_idx]
fam_val_full = fam_val_all.iloc[val_full_idx].reset_index(drop=True)

X_ood = X_test_all.iloc[test_full_idx].reset_index(drop=True)
y_ood = y_test_all[test_full_idx]
fam_ood = fam_test_all.iloc[test_full_idx].reset_index(drop=True)
overall_bar.update(1)

imputer_full = SimpleImputer(strategy="median")
X_train_full_imp = imputer_full.fit_transform(X_train_full)
X_val_full_imp = imputer_full.transform(X_val_full)
X_ood_imp = imputer_full.transform(X_ood)

feat_to_idx = {f: i for i, f in enumerate(feature_cols)}
temporal_idx = [feat_to_idx[f] for f in temporal_cols if f in feat_to_idx]
graph_idx = [feat_to_idx[f] for f in graph_cols if f in feat_to_idx]

if len(temporal_idx) == 0:
    temporal_idx = list(range(min(2, len(feature_cols))))
if len(graph_idx) == 0:
    graph_idx = list(range(min(2, len(feature_cols))))

X_train_temporal = X_train_full_imp[:, temporal_idx]
X_val_temporal = X_val_full_imp[:, temporal_idx]
X_ood_temporal = X_ood_imp[:, temporal_idx]

X_train_graph = X_train_full_imp[:, graph_idx]
X_val_graph = X_val_full_imp[:, graph_idx]
X_ood_graph = X_ood_imp[:, graph_idx]
overall_bar.update(1)

latent_bar = tqdm(total=3, desc="fit full-model branches", position=1, leave=False)

Z_train_temp, temp_transform, info_temp = fit_safe_latent_transformer(
    X_train_temporal, max_comp=6, random_state=SEED, branch_name="temporal"
)
Z_val_temp = temp_transform(X_val_temporal)
Z_ood_temp = temp_transform(X_ood_temporal)
latent_bar.update(1)

Z_train_graph, graph_transform, info_graph = fit_safe_latent_transformer(
    X_train_graph, max_comp=6, random_state=SEED + 1, branch_name="graph"
)
Z_val_graph = graph_transform(X_val_graph)
Z_ood_graph = graph_transform(X_ood_graph)
latent_bar.update(1)

Z_train_ssl, ssl_transform, info_ssl = fit_safe_latent_transformer(
    X_train_full_imp, max_comp=8, random_state=SEED + 2, branch_name="ssl_proxy"
)
Z_val_ssl = ssl_transform(X_val_full_imp)
Z_ood_ssl = ssl_transform(X_ood_imp)
latent_bar.update(1)
latent_bar.close()

model_temp = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED)
model_graph = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 1)
model_ssl = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=120, random_state=SEED + 2)

model_temp.fit(Z_train_temp, y_train_full)
model_graph.fit(Z_train_graph, y_train_full)
model_ssl.fit(Z_train_ssl, y_train_full)

p_val_temp = model_temp.predict_proba(Z_val_temp)[:, 1]
p_ood_temp = model_temp.predict_proba(Z_ood_temp)[:, 1]

p_val_graph = model_graph.predict_proba(Z_val_graph)[:, 1]
p_ood_graph = model_graph.predict_proba(Z_ood_graph)[:, 1]

p_val_ssl = model_ssl.predict_proba(Z_val_ssl)[:, 1]
p_ood_ssl = model_ssl.predict_proba(Z_ood_ssl)[:, 1]

val_fuse_idx, val_rest_idx = train_test_split(
    np.arange(len(y_val_full)),
    test_size=0.60,
    random_state=SEED,
    stratify=y_val_full
)
val_cal_idx, val_thr_idx = train_test_split(
    val_rest_idx,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=y_val_full[val_rest_idx]
)

fusion = LogisticRegression(solver="lbfgs", max_iter=200, random_state=SEED)
fusion.fit(
    np.column_stack([p_val_temp[val_fuse_idx], p_val_graph[val_fuse_idx], p_val_ssl[val_fuse_idx]]),
    y_val_full[val_fuse_idx]
)

p_val_raw = fusion.predict_proba(np.column_stack([p_val_temp, p_val_graph, p_val_ssl]))[:, 1]
p_ood_raw = fusion.predict_proba(np.column_stack([p_ood_temp, p_ood_graph, p_ood_ssl]))[:, 1]

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_val_raw[val_cal_idx], y_val_full[val_cal_idx])

p_val_thr = iso.transform(p_val_raw[val_thr_idx])
p_ood = iso.transform(p_ood_raw)
threshold = choose_best_threshold_macro_f1(y_val_full[val_thr_idx], p_val_thr)
pred_ood = (p_ood >= threshold).astype(np.int8)
conf_ood = np.where(pred_ood == 1, p_ood, 1.0 - p_ood)
overall_bar.update(1)

# ---------------------------------------------------------
# Case selection
# ---------------------------------------------------------
case_meta = pd.DataFrame({
    "y_true": y_ood,
    "y_pred": pred_ood,
    "score_attack": p_ood,
    "confidence": conf_ood,
    "attack_family": fam_ood.astype(str)
})

benign_candidates = case_meta[(case_meta["y_true"] == 0) & (case_meta["y_pred"] == 0)].sort_values("confidence", ascending=False)
attack_candidates = case_meta[(case_meta["y_true"] == 1) & (case_meta["y_pred"] == 1)].sort_values("score_attack", ascending=False)

fn_candidates = case_meta[(case_meta["y_true"] == 1) & (case_meta["y_pred"] == 0)].sort_values("confidence", ascending=False)
fp_candidates = case_meta[(case_meta["y_true"] == 0) & (case_meta["y_pred"] == 1)].sort_values("confidence", ascending=False)
error_candidates = case_meta[(case_meta["y_true"] != case_meta["y_pred"])].sort_values("confidence", ascending=False)

if len(benign_candidates) == 0 or len(attack_candidates) == 0 or len(error_candidates) == 0:
    raise ValueError("Could not find all required case-study examples")

idx_benign = benign_candidates.index[0]
idx_attack = attack_candidates.index[0]
idx_failure = fn_candidates.index[0] if len(fn_candidates) > 0 else error_candidates.index[0]

selected_cases = {
    "Benign correct": idx_benign,
    "Attack correct": idx_attack,
    "Failure case": idx_failure
}

# ---------------------------------------------------------
# Saliency preparation
# ---------------------------------------------------------
scaler_orig = StandardScaler()
X_train_scaled_full = scaler_orig.fit_transform(X_train_full_imp)
X_ood_scaled = scaler_orig.transform(X_ood_imp)

global_weights = compute_feature_weights(X_train_scaled_full, y_train_full)

temporal_feature_names = [feature_cols[i] for i in temporal_idx]
graph_feature_names = [feature_cols[i] for i in graph_idx]

graph_corr = None
if len(graph_idx) >= 2:
    graph_train_scaled = X_train_scaled_full[:, graph_idx]
    graph_corr = np.corrcoef(graph_train_scaled, rowvar=False)
    graph_corr = np.nan_to_num(graph_corr, nan=0.0)
else:
    graph_corr = np.eye(max(1, len(graph_idx)))

# ---------------------------------------------------------
# Figure 9
# ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(2, 3, figsize=(18, 8.5))

case_details_rows = []

for col, (case_name, idx_case) in enumerate(selected_cases.items()):
    x_case = X_ood_scaled[idx_case]
    y_true_case = int(case_meta.loc[idx_case, "y_true"])
    y_pred_case = int(case_meta.loc[idx_case, "y_pred"])
    score_case = float(case_meta.loc[idx_case, "score_attack"])
    conf_case = float(case_meta.loc[idx_case, "confidence"])
    fam_case = str(case_meta.loc[idx_case, "attack_family"])

    local_scores_all = local_feature_scores(x_case, global_weights)

    temporal_scores = local_scores_all[temporal_idx] if len(temporal_idx) > 0 else np.array([])
    graph_scores = local_scores_all[graph_idx] if len(graph_idx) > 0 else np.array([])

    # top temporal features
    if len(temporal_scores) > 0:
        temp_order = np.argsort(-temporal_scores)[:min(5, len(temporal_scores))]
        temp_labels = [temporal_feature_names[i] for i in temp_order]
        temp_vals = temporal_scores[temp_order]
    else:
        temp_labels = ["no_temporal_features"]
        temp_vals = np.array([0.0])

    ax_top = axes[0, col]
    ax_top.barh(temp_labels[::-1], temp_vals[::-1])
    ax_top.set_title(
        f"{case_name}\ntrue={y_true_case}, pred={y_pred_case}, p_attack={score_case:.3f}, family={fam_case}"
    )
    ax_top.set_xlabel("Temporal saliency")
    ax_top.grid(axis="x", alpha=0.25)

    # graph-edge proxy
    ax_bot = axes[1, col]
    if len(graph_scores) > 0:
        edge_df = build_graph_edge_proxy(
            case_graph_scores=graph_scores,
            graph_feature_names=graph_feature_names,
            graph_corr=graph_corr,
            top_k_feats=4,
            top_k_edges=4
        )
        ax_bot.barh(edge_df["edge"].tolist()[::-1], edge_df["score"].tolist()[::-1])
        ax_bot.set_xlabel("Graph-edge saliency (proxy)")
        ax_bot.grid(axis="x", alpha=0.25)
        ax_bot.set_title("Important graph edges")
    else:
        ax_bot.text(0.5, 0.5, "No graph features available", ha="center", va="center")
        ax_bot.set_axis_off()

    case_details_rows.append({
        "case_name": case_name,
        "y_true": y_true_case,
        "y_pred": y_pred_case,
        "attack_probability": score_case,
        "confidence": conf_case,
        "attack_family": fam_case
    })

fig.suptitle("Figure 9. Case Study Explanations for Correct and Incorrect Predictions", y=1.02, fontsize=14)
fig.tight_layout()

fig9_png = os.path.join(OUT_DIR, "figure9_case_study_explanations_for_correct_and_incorrect_predictions.png")
fig9_pdf = os.path.join(OUT_DIR, "figure9_case_study_explanations_for_correct_and_incorrect_predictions.pdf")
fig.savefig(fig9_png, bbox_inches="tight")
fig.savefig(fig9_pdf, bbox_inches="tight")
plt.close(fig)

pd.DataFrame(case_details_rows).to_csv(os.path.join(OUT_DIR, "figure9_selected_case_metadata.csv"), index=False)
overall_bar.update(1)

# ---------------------------------------------------------
# Table 9
# ---------------------------------------------------------
table9_rows = []

# 1) failure case row
failure_row = case_meta.loc[idx_failure]
failure_type = "false_negative" if (failure_row["y_true"] == 1 and failure_row["y_pred"] == 0) else "false_positive"
reason, mitigation = heuristic_reason_and_fix(failure_type=failure_type)
table9_rows.append({
    "error_type": failure_type,
    "attack_family": str(failure_row["attack_family"]),
    "observed_behavior": f"Case prediction was wrong with attack score {failure_row['score_attack']:.3f} and confidence {failure_row['confidence']:.3f}.",
    "likely_reason_for_failure": reason,
    "possible_mitigation": mitigation
})

# 2) benign FP summary
fp_mask = (y_ood == 0) & (pred_ood == 1)
if fp_mask.sum() > 0:
    fp_mean = float(p_ood[fp_mask].mean())
    fp_max = float(p_ood[fp_mask].max())
    reason, mitigation = heuristic_reason_and_fix(failure_type="false_positive")
    table9_rows.append({
        "error_type": "benign_false_positive",
        "attack_family": "benign",
        "observed_behavior": f"{int(fp_mask.sum())} benign OOD samples were flagged as attacks, mean attack score {fp_mean:.3f}, max {fp_max:.3f}.",
        "likely_reason_for_failure": reason,
        "possible_mitigation": mitigation
    })

# 3+) worst families from Table 3
used_families = set([str(failure_row["attack_family"])])

candidate_family_rows = pd.concat([
    table3.sort_values(["recall", "precision", "support"], ascending=[True, True, False]).head(4),
    table3.sort_values(["precision", "recall", "support"], ascending=[True, True, False]).head(4)
], ignore_index=True).drop_duplicates(subset=["attack_family"])

for _, r in candidate_family_rows.iterrows():
    fam = str(r["attack_family"])
    if fam in used_families:
        continue
    precision = float(r["precision"])
    recall = float(r["recall"])
    support = int(r["support"])
    reason, mitigation = heuristic_reason_and_fix(precision=precision, recall=recall)

    if recall < 0.30:
        etype = "family_false_negative_dominant"
    elif precision < 0.30:
        etype = "family_false_positive_prone"
    else:
        etype = "family_partial_overlap"

    table9_rows.append({
        "error_type": etype,
        "attack_family": fam,
        "observed_behavior": f"Precision={precision:.3f}, recall={recall:.3f}, support={support}.",
        "likely_reason_for_failure": reason,
        "possible_mitigation": mitigation
    })
    used_families.add(fam)
    if len(table9_rows) >= 6:
        break

table9 = pd.DataFrame(table9_rows)
table9_path = os.path.join(OUT_DIR, "table9_representative_error_patterns_and_their_causes.csv")
table9.to_csv(table9_path, index=False)
overall_bar.update(1)
overall_bar.close()



In [ ]:
import types

for name, value in globals().items():
    if not name.startswith("_") and not isinstance(value, (types.ModuleType, types.FunctionType, type)):
        try:
            print(f"{name}: {type(value)} | {repr(value)[:150]}")
        except:
            print(f"{name}: {type(value)}")

In [ ]:
import pandas as pd
import types

for name, value in globals().items():
    if name.startswith("_"):
        continue
    if isinstance(value, pd.DataFrame):
        print(f"{name}: DataFrame shape={value.shape}")
    elif not isinstance(value, (types.ModuleType, types.FunctionType, type)):
        print(f"{name}: {type(value)}")

In [ ]:
import numpy as np

# replace df with your actual dataframe name
print("Columns:")
print(df.columns.tolist())
print()

num_scenarios = df["scenario_id"].nunique()
num_unique_vehicles = df["vehicle_id"].nunique()
total_records = len(df)

num_benign = (df["label"] == "benign").sum()
num_attack = total_records - num_benign

num_attack_families = df.loc[df["label"] != "benign", "attack_family"].nunique()

mean_messages_per_scenario = total_records / num_scenarios
mean_active_vehicles_per_scenario = (
    df.groupby("scenario_id")["vehicle_id"].nunique().mean()
)

median_scenario_duration = (
    df.groupby("scenario_id")["time_step"].nunique().median()
)

print("Number of scenarios:", num_scenarios)
print("Number of unique vehicles:", num_unique_vehicles)
print("Total harmonized message records:", total_records)
print("Number of benign records:", num_benign)
print("Number of malicious records:", num_attack)
print("Number of attack families:", num_attack_families)
print("Mean messages per scenario:", round(mean_messages_per_scenario, 2))
print("Mean active vehicles per scenario:", round(mean_active_vehicles_per_scenario, 2))
print("Median scenario duration in time steps:", median_scenario_duration)